# Phase 3 DR calibration, notebook 00 of 04

The cloud design computes alpha-complex persistence once per replication, then uses the cached DR fit for both null calibrations.

Runs shards **0-4**, with 50 total replications. Each completed shard is downloaded immediately, so a disconnected session loses at most the shard in flight.

The stratified-permutation path freezes the cross-fitted nuisances and evaluates all label draws by matrix algebra. It does not rerun PH, cross-fitting, or nuisance regression inside the permutation loop.

In [ ]:
%%bash
set -e
pip install -q numpy scipy scikit-learn matplotlib scikit-fda gudhi ripser persim
pip install -q git+https://github.com/hugogobato/tcda_uq.git
# Colab may start with multimethod 2.1, which conflicts with scikit-fda's
# dispatch metaclass. Pin it last, as in the Phase 2 fleet.
pip install -q 'multimethod==2.0.2'
echo '--- Phase 3 dependencies installed ---'


In [ ]:
import os
for d in ['experiments', 'tda2s', 'tda2s/adapters', 'tda2s/dgp', 'tda2s/ph', 'tda2s/resample', 'tda2s/tests', 'tda2s/vec', 'results/phase3_shards']:
    os.makedirs(d, exist_ok=True)
print('source tree ready')


In [ ]:
%writefile tda2s/__init__.py
"""tda2s: topological two-sample testing infrastructure (shared by P1 and P2)."""

__version__ = "0.0.1"

In [ ]:
%writefile tda2s/adapters/__init__.py
"""Adapters over ``tcda_uq``: P1's import surface for the shared estimators.

See ``docs/reuse_from_tcda_uq.md`` for the full reuse audit and boundary
(CP_TATE owns bands; P1 owns p-values; the shared objects are the AIPW curve
and the per-unit EIF process).
"""

from .tcda_uq import aipw_curve, ctate_learner, silhouettes, tri_oracle

__all__ = ["aipw_curve", "silhouettes", "tri_oracle", "ctate_learner"]


In [ ]:
%writefile tda2s/adapters/tcda_uq.py
"""Thin shim over ``tcda_uq``: import surface for P1, zero reimplemented math.

P1 reuses the released CP_TATE library (``tcda_uq``, installed from git) for
AIPW estimation, cross-fitting, the functional DR-learner, silhouettes, and the
tri-oracle simulation. This module is pure delegation and argument plumbing;
every function forwards to ``tcda_uq`` and only reshapes return values where the
P1 test statistic needs a different layout (e.g. stacking per-dimension score
lists into one array).

Boundary (see ``docs/reuse_from_tcda_uq.md``): tcda_uq owns confidence /
prediction bands; P1 owns p-values. The only shared objects are the AIPW curve
(``aipw[d]``) and the per-unit efficient-influence-function process
(``scores``), from which P1 computes its test statistic and multiplier-bootstrap
null law. Bands are deliberately not exposed here.
"""

from __future__ import annotations

from typing import Tuple

import numpy as np


def aipw_curve(sample, tseq, n_basis: int, n_folds: int = 5, **cross_fit_kwargs) -> dict:
    """Cross-fitted AIPW estimate of the TATE curve(s) for one sample.

    Delegates to ``tcda_uq.estimators.cross_fit`` and reshapes the result for
    P1's test statistic:

    * ``aipw``: list, one ``[resolution]`` mean AIPW curve per homology dim.
    * ``scores``: ``(n, n_hom_dim, resolution)`` per-unit doubly-robust score
      process (the cross-fitted EIF; its mean over units is ``aipw``).
    * ``pi_hat``: ``(n,)`` cross-fitted propensity.
    * ``tseq``: the silhouette grid.

    Args:
        sample: observed triplet ``(phi, A, X)`` with ``phi`` ``[n, n_hom_dim,
            resolution]``, ``A`` ``[n]``, ``X`` ``[n, d]``.
        tseq: silhouette grid ``[resolution]``.
        n_basis: Fourier basis size for the outcome regression.
        n_folds: number of cross-fitting folds (``cross_fit``'s ``n_splits``).
        **cross_fit_kwargs: forwarded verbatim to ``cross_fit`` (e.g.
            ``propensity_estimator``, ``propensity_feature_fn``, ``stratify``,
            ``random_state``); default ``None`` reproduces tcda_uq defaults.
    """
    from tcda_uq.estimators import cross_fit

    result = cross_fit(sample, tseq, n_basis=n_basis, n_splits=n_folds,
                       **cross_fit_kwargs)
    return {
        "aipw": result.aipw,
        "scores": np.stack(result.scores, axis=1),
        "pi_hat": result.pi_hat,
        "tseq": np.asarray(result.tseq),
        # Keep the released result available to P1's calibration layer.  The
        # point estimator and cross-fitting still live entirely in tcda_uq;
        # P1 only uses the fitted fold nuisances to avoid refitting them for
        # every stratified-permutation draw.
        "raw_result": result,
    }


def silhouettes(diagrams, interval=(0.0, 0.2), r: float = 3.0,
                resolution: int = 100) -> np.ndarray:
    """Power-weighted silhouettes of persistence diagrams.

    Delegates to ``tcda_uq.silhouette.compute_silhouette`` (defaults: interval
    ``(0, 0.2)``, ``r=3``, ``resolution=100``). Returns ``(n_hom_dim,
    resolution)``.
    """
    from tcda_uq.silhouette import compute_silhouette

    return compute_silhouette(diagrams, interval=interval, r=r,
                              resolution=resolution)


def tri_oracle(n: int, **kwargs) -> "SimulationSample":
    """Draw a sample from the tri-oracle simulation.

    Delegates to ``tcda_uq.datasets.TriOracleSimulation``: ``kwargs`` are
    passed to its constructor (``n_cov``, ``n_hom_dim``, ``resolution``,
    ``interval``, ``n_basis``, ``noise_scale``, ``seed``, ...), then ``sample(n)``
    is drawn. Returns a ``SimulationSample`` with ``oracle_tate``,
    ``oracle_ctate``, ``oracle_itte`` and the ``.observed`` triplet.
    """
    from tcda_uq.datasets import TriOracleSimulation

    return TriOracleSimulation(**kwargs).sample(n)


def ctate_learner(*args, **kwargs):
    """Functional DR-learner for the CTATE (same signature as ``CTATEDRLearner``).

    Pure delegation to ``tcda_uq.estimators.CTATEDRLearner``; the returned
    object is a ``CTATEDRLearner``: ``fit(sample, tseq, cross_fit_result=None,
    **cross_fit_kwargs)`` then ``predict(X_eval)``.
    """
    from tcda_uq.estimators import CTATEDRLearner

    return CTATEDRLearner(*args, **kwargs)


In [ ]:
%writefile tda2s/resample/__init__.py
"""Resampling engine: null distributions for the Phase 3-5 tests.

Schemes
-------
* ``permutation_test`` -- label permutation (exact null under exchangeability),
  optionally restricted to permutation *within propensity strata* (the
  covariate-preserving variant).
* ``multiplier_bootstrap`` -- Gaussian/Rademacher multiplier bootstrap over a
  per-unit influence-function matrix, calibrated to the weak limit of the
  empirical process: null draws are ``sqrt(n) * sup_t | n^{-1/2} sum_i g_i
  inf_i(t) |``.
* ``paired_bootstrap`` -- unit resampling with replacement preserving the
  treated/control split (paired designs, two-sample mean differences).
* ``smoothed_bootstrap`` -- Roycraft-Krebs-Polonik smoothed bootstrap for
  persistent Betti numbers: resample diagrams with replacement, jitter
  (birth, death) coordinates by N(0, sigma^2), recompute the statistic.
* ``cross_fit_folds`` -- k-fold cross-fitting index splits.
* ``p_value`` -- Monte Carlo p-value helper.
"""
from __future__ import annotations

from typing import Callable, List, Optional, Sequence, Tuple

import numpy as np

from .smoothing import betti_curve

__all__ = [
    "permutation_test",
    "multiplier_bootstrap",
    "paired_bootstrap",
    "smoothed_bootstrap",
    "cross_fit_folds",
    "p_value",
]


def p_value(observed: float, null_stats: Sequence[float], alternative: str = "greater") -> float:
    """Monte Carlo p-value with Phipson-Smyth correction.

    ``alternative="greater"``: P(T* >= T_obs); ``"less"``: P(T* <= T_obs);
    ``"two-sided"``: P(|T*| >= |T_obs|) with the same correction.
    """
    null_stats = np.asarray(null_stats, dtype=float)
    if null_stats.size == 0:
        return 1.0
    if alternative == "greater":
        return (1.0 + (null_stats >= observed).sum()) / (1.0 + null_stats.size)
    if alternative == "less":
        return (1.0 + (null_stats <= observed).sum()) / (1.0 + null_stats.size)
    if alternative == "two-sided":
        return (1.0 + (np.abs(null_stats) >= abs(observed)).sum()) / (1.0 + null_stats.size)
    raise ValueError(f"unknown alternative: {alternative}")


def permutation_test(stat_fn: Callable, group_labels: np.ndarray, n_perm: int,
                     rng: np.random.Generator,
                     strata: Optional[np.ndarray] = None) -> Tuple[float, np.ndarray]:
    """Label-permutation test of ``stat_fn``.

    Args:
        stat_fn: callable taking ``(labels)`` and returning the statistic.
            The observed statistic is ``stat_fn(group_labels)``.
        group_labels: length-n binary treatment labels.
        n_perm: number of permutations.
        rng: numpy Generator.
        strata: optional length-n stratum ids; labels are permuted only within
            each stratum (covariate-preserving null).

    Returns:
        ``(observed_stat, null_stats)``.
    """
    labels = np.asarray(group_labels).copy()
    n = labels.size
    null_stats = np.empty(n_perm)
    observed = stat_fn(labels)

    if strata is None:
        for b in range(n_perm):
            null_stats[b] = stat_fn(labels[rng.permutation(n)])
    else:
        strata = np.asarray(strata)
        permuted = labels.copy()
        for s in np.unique(strata):
            idx = np.flatnonzero(strata == s)
            permuted[idx] = labels[idx][rng.permutation(idx.size)]
        for b in range(n_perm):
            for s in np.unique(strata):
                idx = np.flatnonzero(strata == s)
                permuted[idx] = labels[idx][rng.permutation(idx.size)]
            null_stats[b] = stat_fn(permuted)
    return float(observed), null_stats


def multiplier_bootstrap(influence_matrix: np.ndarray, n_draws: int,
                         rng: np.random.Generator,
                         kind: str = "gaussian") -> np.ndarray:
    """Multiplier bootstrap over a per-unit influence matrix.

    Args:
        influence_matrix: ``(n, resolution)`` per-unit influence-function
            values (row = unit). Convention matches tcda_uq's ``scores``.
        n_draws: number of null draws.
        rng: numpy Generator.
        kind: multiplier law: "gaussian" (N(0,1)) or "rademacher" (+-1).

    Returns:
        ``n_draws`` null statistics ``sup_t | n^{-1/2} sum_i g_i inf_i(t) |``.
        This is on the same scale as ``sqrt(n) * sup_t |mean_i inf_i(t)|``, the
        studentised statistic ``T_n`` of the plan, so observed values may be
        compared with these draws directly.
    """
    inf = np.asarray(influence_matrix, dtype=float)
    n = inf.shape[0]
    n_res = inf.shape[1]
    null_stats = np.empty(n_draws)
    scaled = inf / np.sqrt(n)
    for b in range(n_draws):
        if kind == "gaussian":
            g = rng.standard_normal(n)
        elif kind == "rademacher":
            g = rng.choice([-1.0, 1.0], size=n)
        else:
            raise ValueError(f"unknown multiplier kind: {kind}")
        curve = g @ scaled
        null_stats[b] = np.max(np.abs(curve))
    return null_stats


def paired_bootstrap(stat_fn: Callable, group_labels: np.ndarray, n_draws: int,
                     rng: np.random.Generator) -> Tuple[float, np.ndarray]:
    """Unit resampling with replacement preserving the treated/control split.

    Args:
        stat_fn: callable taking ``(labels)``.
        group_labels: length-n binary labels.
        n_draws: number of bootstrap draws.

    Returns:
        ``(observed_stat, bootstrap_stats)``.
    """
    labels = np.asarray(group_labels)
    n = labels.size
    n1 = int((labels == 1).sum())
    observed = stat_fn(labels)
    null_stats = np.empty(n_draws)
    for b in range(n_draws):
        idx = np.concatenate([
            rng.choice(np.flatnonzero(labels == 1), size=n1, replace=True),
            rng.choice(np.flatnonzero(labels == 0), size=n - n1, replace=True),
        ])
        null_stats[b] = stat_fn(labels[idx])
    return float(observed), null_stats


def smoothed_bootstrap(diagrams: Sequence[Sequence[np.ndarray]], n_draws: int,
                       rng: np.random.Generator, sigma: float,
                       stat_fn: Callable) -> Tuple[float, np.ndarray]:
    """Smoothed (jittered) bootstrap for persistent Betti numbers.

    Roycraft-Krebs-Polonik: the naive bootstrap is inconsistent for persistent
    Betti numbers; jittering (birth, death) coordinates by N(0, sigma^2) fixes
    the boundary effects.

    Args:
        diagrams: list over samples of list of (k, 2) per-dim diagrams.
        n_draws: number of bootstrap samples.
        rng: numpy Generator.
        sigma: jitter bandwidth (e.g. bandwidth / 2 of the kernel density).
        stat_fn: callable taking a list of diagram-lists (one per sample) and
            returning the statistic.

    Returns:
        ``(observed_stat, bootstrap_stats)``.
    """
    diagrams = [[np.asarray(d, dtype=float).reshape(-1, 2) for d in per_dim]
                for per_dim in diagrams]
    observed = stat_fn(diagrams)
    null_stats = np.empty(n_draws)
    n_samples = len(diagrams)
    for b in range(n_draws):
        # Resample WHOLE samples (diagram lists) with replacement -- one drawn
        # index per bootstrap unit. Pooling several units into a single diagram
        # would multiply every feature count by the pool size.
        idx = rng.integers(0, n_samples, size=n_samples)
        resampled = []
        for i in idx:
            per_dim = []
            for dgm in diagrams[i]:
                if dgm.size == 0:
                    per_dim.append(np.zeros((0, 2)))
                    continue
                jittered = dgm + rng.normal(0.0, sigma, size=dgm.shape)
                # keep the diagram above the diagonal after jittering
                jittered[:, 1] = np.maximum(jittered[:, 1], jittered[:, 0])
                per_dim.append(jittered)
            resampled.append(per_dim)
        null_stats[b] = stat_fn(resampled)
    return float(observed), null_stats


def cross_fit_folds(n: int, k_folds: int, rng: np.random.Generator,
                    stratify_labels: Optional[np.ndarray] = None) -> List[Tuple[np.ndarray, np.ndarray]]:
    """k-fold cross-fitting index splits.

    Returns:
        List of ``(train_idx, test_idx)`` pairs covering all n indices exactly
        once as test indices. If ``stratify_labels`` is given, class balance is
        preserved within folds.
    """
    if stratify_labels is None:
        perm = rng.permutation(n)
        fold_of = np.zeros(n, dtype=int)
        for f in range(k_folds):
            fold_of[perm[f::k_folds]] = f
    else:
        labels = np.asarray(stratify_labels)
        fold_of = np.empty(n, dtype=int)
        for lab in np.unique(labels):
            idx = np.flatnonzero(labels == lab)
            perm = rng.permutation(idx.size)
            for f in range(k_folds):
                fold_of[idx[perm[f::k_folds]]] = f
    folds = []
    for f in range(k_folds):
        test_idx = np.flatnonzero(fold_of == f)
        train_idx = np.flatnonzero(fold_of != f)
        folds.append((train_idx, test_idx))
    return folds

In [ ]:
%writefile tda2s/resample/smoothing.py
"""Betti-curve helpers for the smoothed bootstrap (Roycraft-Krebs-Polonik).

Re-exports the canonical ``betti_curve`` from ``tda2s.vec`` and adds the
sample-level aggregate used by smoothed-bootstrap statistics.
"""
from __future__ import annotations

import numpy as np

from tda2s.vec import betti_curve as _vec_betti_curve


def _max_death(diagrams):
    """Largest finite death across a list of per-dim (k, 2) arrays."""
    m = 0.0
    for dgm in diagrams:
        dgm = np.asarray(dgm, dtype=float)
        if dgm.ndim == 2 and dgm.size:
            finite = dgm[np.isfinite(dgm[:, 1]), 1]
            if finite.size:
                m = max(m, float(finite.max()))
    return m


def betti_curve(diagrams, interval=None, n_points=100):
    """Persistent Betti-number curve of ONE sample's diagram list.

    Args:
        diagrams: list of (k, 2) per-dim (birth, death) arrays (one sample).
        interval: (t_min, t_max) grid; defaults to (0, max death).
        n_points: grid resolution.

    Returns:
        ``(t_grid, betti_matrix)`` with ``betti_matrix[d]`` the B_d(t) curve.
    """
    iv = interval if interval is not None else (0.0, _max_death(diagrams) + 1e-9)
    grid = np.linspace(iv[0], iv[1], n_points)
    matrix = _vec_betti_curve(diagrams, interval=iv, n_points=n_points)
    return grid, matrix


def mean_betti_curve(sample_diagrams, interval=None, n_points=100):
    """Mean Betti curve over a sample of diagrams (for bootstrap statistics).

    All samples are evaluated on ONE shared grid. When ``interval`` is not
    given it is derived from the *pooled* diagrams, not per sample: averaging
    curves that were each sampled on their own grid would mix incomparable
    abscissae and silently distort the mean.

    Args:
        sample_diagrams: list over samples of list of (k, 2) per-dim arrays.
        interval: shared (t_min, t_max); defaults to (0, pooled max death).
        n_points: grid resolution.

    Returns:
        ``(t_grid, mean_betti)`` with ``mean_betti[d]`` a length-``n_points``
        curve averaged over the sample.
    """
    if interval is None:
        pooled = 0.0
        for diags in sample_diagrams:
            pooled = max(pooled, _max_death(diags))
        interval = (0.0, pooled + 1e-9) if pooled > 0 else (0.0, 1.0)
    grid = np.linspace(interval[0], interval[1], n_points)
    if not sample_diagrams:
        return grid, np.zeros((1, n_points))
    curves = [_vec_betti_curve(diags, interval=interval, n_points=n_points)
              for diags in sample_diagrams]
    return grid, np.mean(np.stack(curves), axis=0)

In [ ]:
%writefile tda2s/tests/__init__.py
"""Outcome-level DR testing utilities used by the Phase 3 experiments."""



In [ ]:
%writefile tda2s/tests/dr_outcome.py
"""Calibrated outcome-level doubly robust tests for Phase 3.

The AIPW point estimate and cross-fitting are delegated to ``tcda_uq``.  This
module adds the testing layer that P1 owns:

* the max-over-scale and max-over-degree statistic;
* a shared-multiplier null over the cross-fitted score process;
* a fast covariate-preserving permutation null.

The permutation implementation is deliberately explicit about its scope.  A
full permutation test that refits the nuisances for every label draw is
prohibitively expensive.  ``fit_dr`` fits cross-fitted nuisances once,
reconstructs their out-of-fold predictions in original sample order, and
``stratified_permutation_test`` evaluates the AIPW score with those predictions
held fixed.  This is exact conditional randomisation inference when the fitted
nuisances and propensity strata are fixed independently of the permuted labels,
for example under a sharp conditional null with externally fixed design
information.  With estimated, label-dependent nuisances it is a fast
cross-fitted calibration approximation, not a finite-sample exact theorem.
The distinction is part of the Phase 3 report and is not hidden by the API.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Callable, Iterable, Optional

import numpy as np
from sklearn.ensemble import (HistGradientBoostingClassifier,
                               RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold, StratifiedKFold

from tda2s.adapters.tcda_uq import aipw_curve
from tda2s.resample import multiplier_bootstrap, p_value


def _clip_propensity(pi: np.ndarray) -> np.ndarray:
    """Match tcda_uq's propensity clipping convention."""
    out = np.asarray(pi, dtype=float).copy()
    out[out <= 0.0] = 1e-2
    out[out >= 1.0] = 1.0 - 1e-2
    return out


def _split_test_indices(A: np.ndarray, X: np.ndarray, n_folds: int,
                        stratify: bool, random_state: Optional[int]):
    """Reproduce the test-fold order used by tcda_uq.cross_fit."""
    if stratify:
        splitter = StratifiedKFold(n_splits=n_folds, shuffle=True,
                                   random_state=random_state)
        iterator = splitter.split(X, A)
    else:
        splitter = KFold(n_splits=n_folds, shuffle=True,
                         random_state=random_state)
        iterator = splitter.split(X)
    return list(iterator)


def _as_score_tensor(values: Iterable[np.ndarray]) -> np.ndarray:
    """Stack tcda_uq's per-dimension curves as ``[n, d, t]``."""
    return np.stack([np.asarray(v, dtype=float) for v in values], axis=1)


@dataclass
class DRFit:
    """Cached cross-fitted nuisances and data for Phase 3 calibration."""

    phi: np.ndarray                 # original order, [n, d, t]
    A: np.ndarray                   # original order, [n]
    X: np.ndarray                   # original order, [n, p]
    tseq: np.ndarray
    result: Any                     # tcda_uq CrossFitResult
    order: np.ndarray               # score/fold order -> original row index
    labels_order: np.ndarray        # labels in score/fold order
    phi_order: np.ndarray           # outcomes in score/fold order
    mu0: np.ndarray                 # fixed out-of-fold predictions [n, d, t]
    mu1: np.ndarray                 # fixed out-of-fold predictions [n, d, t]
    pi_hat: np.ndarray              # fixed out-of-fold propensities [n]
    fold_ids: np.ndarray            # fold id in score/fold order
    n_basis: int
    n_folds: int
    stratify: bool
    random_state: Optional[int]
    propensity_feature_fn: Optional[Callable]

    @property
    def n(self) -> int:
        return int(self.phi.shape[0])

    @property
    def n_hom_dim(self) -> int:
        return int(self.phi.shape[1])

    @property
    def resolution(self) -> int:
        return int(self.phi.shape[2])

    @property
    def estimate(self) -> np.ndarray:
        """AIPW curve in dimension-major shape ``[d, t]``."""
        return _as_score_tensor(self.result.scores).mean(axis=0)

    @property
    def centered_scores(self) -> np.ndarray:
        """Centered cross-fitted score process in score/fold order."""
        return _as_score_tensor(self.result.scores) - self.estimate[None, :, :]


def fit_dr(sample, tseq, *, n_basis: int = 8, n_folds: int = 2,
           propensity_estimator=None, stratify: bool = True,
           random_state: Optional[int] = 0,
           propensity_feature_fn: Optional[Callable] = None) -> DRFit:
    """Fit tcda_uq's cross-fitted AIPW estimator and cache fold predictions.

    No estimator is reimplemented here.  The fold reconstruction is only an
    adapter around the public ``CrossFitResult.folds`` objects, allowing the
    permutation layer to evaluate many label assignments without refitting.
    """
    phi, A, X = (np.asarray(sample[0]), np.asarray(sample[1]),
                 np.asarray(sample[2]))
    if phi.ndim != 3 or A.ndim != 1 or X.ndim != 2:
        raise ValueError("sample must have phi[n,d,t], A[n], and X[n,p]")
    if len(A) != len(phi) or len(X) != len(phi):
        raise ValueError("phi, A, and X must have the same number of rows")
    A = A.astype(int, copy=False)
    if not np.all(np.isin(A, (0, 1))):
        raise ValueError("A must contain only 0/1 labels")

    wrapped = aipw_curve(
        (phi, A, X), np.asarray(tseq), n_basis=n_basis, n_folds=n_folds,
        propensity_estimator=propensity_estimator, stratify=stratify,
        random_state=random_state, propensity_feature_fn=propensity_feature_fn,
    )
    result = wrapped["raw_result"]
    folds = _split_test_indices(A, X, n_folds, stratify, random_state)
    order = np.concatenate([test_idx for _, test_idx in folds])
    if not np.array_equal(order, np.asarray(result.order)):
        raise RuntimeError("could not reproduce tcda_uq cross-fit fold order")
    if len(folds) != len(result.folds):
        raise RuntimeError("tcda_uq returned an unexpected number of folds")

    n, n_dim, res = phi.shape
    mu0 = np.empty((n, n_dim, res), dtype=float)
    mu1 = np.empty_like(mu0)
    pi_parts = []
    fold_id_parts = []
    for fold_id, ((_, test_idx), fitted) in enumerate(zip(folds, result.folds)):
        mu_hats = fitted.predict_mu(X[test_idx])
        if len(mu_hats) != n_dim:
            raise RuntimeError("tcda_uq returned an unexpected dimension count")
        for d, (m0, m1) in enumerate(mu_hats):
            mu0[test_idx, d, :] = np.asarray(m0, dtype=float)
            mu1[test_idx, d, :] = np.asarray(m1, dtype=float)
        features = (propensity_feature_fn(X[test_idx])
                    if propensity_feature_fn is not None else X[test_idx])
        pi_parts.append(np.asarray(
            fitted.prop_model.predict_proba(features)[:, 1], dtype=float))
        fold_id_parts.append(np.full(len(test_idx), fold_id, dtype=int))

    # ``result.scores`` is concatenated in fold order, whereas the nuisance
    # arrays above are assigned in original order.  Put all cached quantities
    # into one unambiguous order for fast matrix calculations.
    pi_order = _clip_propensity(np.concatenate(pi_parts))
    fold_ids = np.concatenate(fold_id_parts)
    return DRFit(
        phi=phi, A=A, X=X, tseq=np.asarray(tseq), result=result,
        order=order, labels_order=A[order], phi_order=phi[order],
        mu0=mu0[order], mu1=mu1[order], pi_hat=pi_order,
        fold_ids=fold_ids, n_basis=n_basis, n_folds=n_folds,
        stratify=stratify, random_state=random_state,
        propensity_feature_fn=propensity_feature_fn,
    )


def _scores_for_labels(fit: DRFit, labels_order: np.ndarray) -> np.ndarray:
    """Evaluate the AIPW score using cached nuisances and new labels."""
    labels = np.asarray(labels_order, dtype=float)
    if labels.shape != (fit.n,):
        raise ValueError("labels must have one entry per fitted unit")
    if not np.all(np.isin(labels, (0.0, 1.0))):
        raise ValueError("labels must contain only 0/1 values")
    inv_treat = (labels / fit.pi_hat)[:, None, None]
    inv_control = ((1.0 - labels) / (1.0 - fit.pi_hat))[:, None, None]
    return (fit.mu1 - fit.mu0 + inv_treat * (fit.phi_order - fit.mu1)
            - inv_control * (fit.phi_order - fit.mu0))


def _select_curves(curves: np.ndarray, degrees=None) -> np.ndarray:
    curves = np.asarray(curves, dtype=float)
    if curves.ndim != 2:
        raise ValueError("curves must have shape [dimension, resolution]")
    if degrees is None:
        return curves
    idx = np.asarray(list(degrees), dtype=int)
    if idx.ndim != 1 or len(idx) == 0 or np.any(idx < 0) or np.any(idx >= len(curves)):
        raise ValueError("degrees must be valid non-empty dimension indices")
    return curves[idx]


def _curve_norm(curves: np.ndarray, norm: str) -> np.ndarray:
    """Return one max-over-degree norm for each leading draw."""
    curves = np.asarray(curves, dtype=float)
    if norm == "sup":
        return np.max(np.abs(curves), axis=(-2, -1))
    if norm == "l2":
        # Root-mean-square on the common grid.  The common 1/resolution factor
        # makes this comparable across the Phase 3 grids.
        return np.max(np.sqrt(np.mean(curves ** 2, axis=-1)), axis=-1)
    raise ValueError("norm must be 'sup' or 'l2'")


def dr_statistic(curves: np.ndarray, n: int, *, degrees=None,
                 studentize: bool = False, score_values: Optional[np.ndarray] = None,
                 sd_floor: float = 1e-10, norm: str = "sup") -> float:
    """Compute ``sqrt(n) max_d sup_t |psi_hat_d(t)|``.

    If ``studentize=True``, ``score_values`` must contain the per-unit score
    process in the same order as ``curves`` and the statistic uses its
    pointwise empirical standard deviation.  The raw statistic is the primary
    Phase 3 specification; studentization is a diagnostic/robustness option.
    """
    selected = _select_curves(curves, degrees)
    if studentize:
        if score_values is None:
            raise ValueError("score_values required for studentization")
        scores = np.asarray(score_values, dtype=float)
        sd = scores.std(axis=0, ddof=1)
        sd = np.maximum(_select_curves(sd, degrees), sd_floor)
        selected = selected / sd
    return float(np.sqrt(n) * _curve_norm(selected[None, :, :], norm)[0])


def _bootstrap_stats(centered: np.ndarray, n_draws: int, rng: np.random.Generator,
                     *, degrees=None, studentize: bool = False,
                     sd_floor: float = 1e-10, multiplier: str = "gaussian",
                     norm: str = "sup") -> np.ndarray:
    """Shared-multiplier max statistics with one multiplier per unit."""
    centered = np.asarray(centered, dtype=float)
    n, n_dim, res = centered.shape
    if not studentize and norm == "sup":
        flat = centered.reshape(n, n_dim * res)
        return multiplier_bootstrap(flat, n_draws, rng, kind=multiplier)

    sd = centered.std(axis=0, ddof=1) if studentize else None
    if sd is not None:
        sd = np.maximum(_select_curves(sd, degrees), sd_floor)
    d_idx = list(range(n_dim)) if degrees is None else list(np.asarray(list(degrees), dtype=int))
    out = np.empty(n_draws, dtype=float)
    for b in range(n_draws):
        if multiplier == "gaussian":
            xi = rng.standard_normal(n)
        elif multiplier == "rademacher":
            xi = rng.choice(np.array([-1.0, 1.0]), size=n)
        else:
            raise ValueError("multiplier must be 'gaussian' or 'rademacher'")
        draw = (xi[:, None, None] * centered).sum(axis=0) / np.sqrt(n)
        if studentize:
            draw = draw[d_idx] / sd
        else:
            draw = draw[d_idx]
        out[b] = _curve_norm(draw[None, :, :], norm)[0]
    return out


def multiplier_test(fit: DRFit, *, n_draws: int = 2000, alpha: float = 0.05,
                    degrees=None, studentize: bool = False, multiplier: str = "gaussian",
                    seed: Optional[int] = 0, norm: str = "sup") -> dict:
    """Calibrate the outcome-level test by a shared multiplier process."""
    observed_scores = _scores_for_labels(fit, fit.labels_order)
    observed = observed_scores.mean(axis=0)
    centered = observed_scores - observed[None, :, :]
    stat = dr_statistic(observed, fit.n, degrees=degrees, studentize=studentize,
                        score_values=observed_scores, norm=norm)
    null = _bootstrap_stats(centered, n_draws, np.random.default_rng(seed),
                            degrees=degrees, studentize=studentize,
                            multiplier=multiplier, norm=norm)
    return {
        "method": "multiplier",
        "statistic": stat,
        "pvalue": p_value(stat, null),
        "null": null,
        "critical_value": float(np.quantile(null, 1.0 - alpha)),
        "estimate": observed,
        "scores": observed_scores,
    }


def degree_multiplicity_test(fit: DRFit, *, n_draws: int = 2000,
                             alpha: float = 0.05, studentize: bool = False,
                             multiplier: str = "gaussian",
                             seed: Optional[int] = 0) -> dict:
    """Bonferroni and max-statistic calibration across homology degrees.

    The max-statistic uses one multiplier per unit shared across all degrees,
    preserving their dependence.  The Bonferroni line is included as the
    transparent conservative comparator.  A Vejdemo-Johansson--Mukherjee
    implementation is not silently substituted here: its published
    persistence-specific multiple-testing construction remains a separate
    benchmark to add once its exact finite-sample convention is frozen.
    """
    per_degree = [
        multiplier_test(fit, n_draws=n_draws, alpha=alpha, degrees=[d],
                        studentize=studentize, multiplier=multiplier,
                        seed=seed)
        for d in range(fit.n_hom_dim)
    ]
    max_test = multiplier_test(
        fit, n_draws=n_draws, alpha=alpha, degrees=range(fit.n_hom_dim),
        studentize=studentize, multiplier=multiplier, seed=seed,
    )
    raw_p = min((entry["pvalue"] for entry in per_degree), default=1.0)
    return {
        "per_degree": per_degree,
        "bonferroni_pvalue": float(min(1.0, fit.n_hom_dim * raw_p)),
        "max_statistic": max_test,
        "max_statistic_pvalue": float(max_test["pvalue"]),
    }


def propensity_strata(pi_hat: np.ndarray, n_bins: int = 5) -> np.ndarray:
    """Create deterministic quantile propensity strata for diagnostics.

    Quantile bins preserve propensity approximately.  For an exact conditional
    randomisation claim, pass externally fixed strata to
    :func:`stratified_permutation_test` instead.
    """
    pi = np.asarray(pi_hat, dtype=float)
    if pi.ndim != 1 or len(pi) == 0 or n_bins < 1:
        raise ValueError("pi_hat must be non-empty 1-D and n_bins >= 1")
    edges = np.unique(np.quantile(pi, np.linspace(0.0, 1.0, n_bins + 1)))
    if len(edges) <= 1:
        return np.zeros(len(pi), dtype=int)
    return np.digitize(pi, edges[1:-1], right=True).astype(int)


def _draw_stratified_labels(labels: np.ndarray, strata: np.ndarray,
                            n_draws: int, rng: np.random.Generator) -> np.ndarray:
    labels = np.asarray(labels, dtype=float)
    strata = np.asarray(strata)
    if labels.shape != strata.shape:
        raise ValueError("labels and strata must have the same shape")
    draws = np.broadcast_to(labels, (n_draws, len(labels))).copy()
    for s in np.unique(strata):
        idx = np.flatnonzero(strata == s)
        for b in range(n_draws):
            draws[b, idx] = labels[idx][rng.permutation(len(idx))]
    return draws


def _permutation_stats(fit: DRFit, label_draws: np.ndarray, *, degrees=None,
                       studentize: bool = False, batch_size: int = 64,
                       sd_floor: float = 1e-10, norm: str = "sup") -> np.ndarray:
    """Vectorized permutation statistics with a bounded temporary footprint."""
    draws = np.asarray(label_draws, dtype=float)
    if draws.ndim != 2 or draws.shape[1] != fit.n:
        raise ValueError("label_draws must have shape [n_draws, n]")
    n_draws = draws.shape[0]
    base = fit.mu1 - fit.mu0
    residual1 = fit.phi_order - fit.mu1
    residual0 = fit.phi_order - fit.mu0
    base_mean = base.mean(axis=0)
    inv_pi = 1.0 / fit.pi_hat
    inv_one_minus_pi = 1.0 / (1.0 - fit.pi_hat)
    if studentize:
        observed_scores = _scores_for_labels(fit, fit.labels_order)
        sd = observed_scores.std(axis=0, ddof=1)
        sd = np.maximum(_select_curves(sd, degrees), sd_floor)
    d_idx = list(range(fit.n_hom_dim)) if degrees is None else list(np.asarray(list(degrees), dtype=int))
    out = np.empty(n_draws, dtype=float)
    for lo in range(0, n_draws, batch_size):
        hi = min(lo + batch_size, n_draws)
        a = draws[lo:hi]
        treated = np.einsum("bn,nhr->bhr", a * inv_pi, residual1)
        control = np.einsum("bn,nhr->bhr", (1.0 - a) * inv_one_minus_pi, residual0)
        curves = base_mean[None, :, :] + (treated - control) / fit.n
        if studentize:
            curves = curves[:, d_idx, :] / sd[None, :, :]
        else:
            curves = curves[:, d_idx, :]
        out[lo:hi] = _curve_norm(curves, norm) * np.sqrt(fit.n)
    return out


def stratified_permutation_test(fit: DRFit, strata: np.ndarray, *, n_perm: int = 999,
                                alpha: float = 0.05, degrees=None,
                                studentize: bool = False, seed: Optional[int] = 0,
                                batch_size: int = 64, norm: str = "sup") -> dict:
    """Fast covariate-preserving permutation calibration.

    ``strata`` is supplied in original sample order.  Labels are permuted only
    within each stratum, while the cached cross-fitted nuisances are held fixed.
    No persistent homology, regression, or cross-fitting is performed inside
    the permutation loop.
    """
    strata = np.asarray(strata)
    if strata.shape != (fit.n,):
        raise ValueError("strata must be in original sample order with length n")
    strata_order = strata[fit.order]
    observed_scores = _scores_for_labels(fit, fit.labels_order)
    observed = observed_scores.mean(axis=0)
    stat = dr_statistic(observed, fit.n, degrees=degrees, studentize=studentize,
                        score_values=observed_scores, norm=norm)
    rng = np.random.default_rng(seed)
    label_draws = _draw_stratified_labels(fit.labels_order, strata_order,
                                          n_perm, rng)
    null = _permutation_stats(fit, label_draws, degrees=degrees,
                              studentize=studentize, batch_size=batch_size,
                              norm=norm)
    return {
        "method": "stratified_permutation_frozen_nuisance",
        "statistic": stat,
        "pvalue": p_value(stat, null),
        "null": null,
        "critical_value": float(np.quantile(null, 1.0 - alpha)),
        "estimate": observed,
        "scores": observed_scores,
        "strata": strata,
        "n_strata": int(len(np.unique(strata))),
    }


def positivity_diagnostics(fit: DRFit) -> dict:
    """Overlap and effective-sample-size diagnostics for a fitted DR test."""
    pi = fit.pi_hat
    A = fit.labels_order.astype(bool)
    wt1 = A / pi
    wt0 = (~A) / (1.0 - pi)

    def ess(weights):
        weights = weights[weights > 0]
        return float(weights.sum() ** 2 / np.sum(weights ** 2)) if len(weights) else 0.0

    return {
        "min_pi": float(pi.min()),
        "max_pi": float(pi.max()),
        "q01_pi": float(np.quantile(pi, 0.01)),
        "q99_pi": float(np.quantile(pi, 0.99)),
        "fraction_pi_at_clip": float(np.mean((pi <= 0.0100001) | (pi >= 0.9899999))),
        "n_treated": int(A.sum()),
        "n_control": int((~A).sum()),
        "ess_treated": ess(wt1),
        "ess_control": ess(wt0),
    }


def equivalence_test(fit: DRFit, margin: float, *, alpha: float = 0.05,
                     n_draws: int = 2000, degrees=None, multiplier: str = "gaussian",
                     seed: Optional[int] = 0) -> dict:
    """Sup-norm TOST-style equivalence test for ``||psi||_inf < margin``.

    The test rejects the non-equivalence null when the simultaneous upper
    confidence bound for the selected max norm is at most ``margin``.  This is
    the functional analogue of the two one-sided tests and controls degree
    multiplicity through the selected max statistic.
    """
    observed_scores = _scores_for_labels(fit, fit.labels_order)
    estimate = observed_scores.mean(axis=0)
    centered = observed_scores - estimate[None, :, :]
    selected = _select_curves(estimate, degrees)
    observed_norm = float(np.max(np.abs(selected)))
    errors = _bootstrap_stats(centered, n_draws, np.random.default_rng(seed),
                              degrees=degrees, multiplier=multiplier)
    q = float(np.quantile(errors, 1.0 - alpha)) / np.sqrt(fit.n)
    threshold = np.sqrt(fit.n) * (margin - observed_norm)
    p_equiv = p_value(threshold, errors, alternative="greater")
    return {
        "method": "supnorm_equivalence",
        "margin": float(margin),
        "observed_norm": observed_norm,
        "upper_bound": observed_norm + q,
        "pvalue": p_equiv,
        "reject_non_equivalence": bool(observed_norm + q <= margin),
        "null_error": errors,
    }


def propensity_learner_grid(seed: int = 0) -> dict:
    """Small, reproducible propensity learner grid for task 3.3."""
    return {
        "logistic": LogisticRegression(max_iter=2000, random_state=seed),
        "random_forest": RandomForestClassifier(
            n_estimators=200, min_samples_leaf=5, n_jobs=1, random_state=seed),
        "gradient_boosting": HistGradientBoostingClassifier(
            max_iter=150, learning_rate=0.05, max_leaf_nodes=15,
            random_state=seed),
        "neural": MLPClassifier(hidden_layer_sizes=(32,), max_iter=400,
                                early_stopping=True, random_state=seed),
    }


In [ ]:
%writefile tda2s/dgp/__init__.py
"""Controlled DGP harness: point-cloud generators and two-group datasets.

Public API:
  * generators: ``circle_cloud``, ``torus_cloud``, ``sphere_cloud``,
    ``cluster_cloud``, ``loops_cloud``, ``split_cluster_cloud`` (the WP1.1 /
    Phase 4.4 cluster-splitting witness);
  * harness: ``CloudSampleDGP`` (two-group datasets with covariate-driven
    topology, propensity, and group-effect knobs), ``CloudSample`` (the
    realised dataset with per-cloud oracles), ``masking_stratum_sample``
    (the Phase 2.2 exact Simpson-masking DGP: H0^cond true, H0^out false);
  * export: ``to_silhouette_sample`` (clouds -> tcda_uq-format ``(phi, A, X)``).
"""

from .clouds import (circle_cloud, cluster_cloud, loops_cloud, sphere_cloud,
                     split_cluster_cloud, torus_cloud)
from .simulation import (CloudSample, CloudSampleDGP, masking_stratum_sample,
                         to_silhouette_sample)

__all__ = [
    "circle_cloud",
    "torus_cloud",
    "sphere_cloud",
    "cluster_cloud",
    "loops_cloud",
    "split_cluster_cloud",
    "CloudSampleDGP",
    "CloudSample",
    "masking_stratum_sample",
    "to_silhouette_sample",
]

In [ ]:
%writefile tda2s/dgp/clouds.py
"""Controlled point-cloud generators (Phase 0.6).

Basic shape generators (circle, torus, sphere, cluster) plus the loop cloud
used by the covariate-driven DGP harness: ``n_loops`` circles arranged on a big
circle, each with its own radius, dialable noise and outlier fraction.

All generators return ``(n, d)`` float arrays (2-D for circle / cluster /
loops, 3-D for torus / sphere) and accept an ``rng`` argument that may be an
integer seed, ``None``, or a ``numpy`` Generator (a Generator is used as-is so
streams stay composable).

Memory discipline: generators are vectorised and allocate only ``O(n)``; they
are intended for small clouds (``n <= 300``).
"""

from __future__ import annotations

import numpy as np


def _as_rng(rng):
    """Normalise ``rng`` to a ``numpy`` Generator (Generator passthrough)."""
    return rng if isinstance(rng, np.random.Generator) else np.random.default_rng(rng)


def circle_cloud(n, radius=1.0, noise=0.05, rng=None):
    """Noisy circle: ``n`` points on a circle of radius ``radius`` plus Gaussian jitter.

    Args:
        n: number of points.
        radius: circle radius (the persistent ``H_1`` feature scale).
        noise: standard deviation of the isotropic Gaussian jitter.
        rng: seed or Generator.

    Returns:
        ``(n, 2)`` point cloud with one prominent ``H_1`` feature of persistence
        approximately ``radius``.
    """
    rng = _as_rng(rng)
    theta = rng.uniform(0.0, 2.0 * np.pi, size=n)
    pts = np.column_stack([radius * np.cos(theta), radius * np.sin(theta)])
    return pts + rng.normal(scale=noise, size=pts.shape)


def torus_cloud(n, R=2.0, r=0.6, noise=0.05, rng=None):
    """Noisy torus: ``n`` points on a 3-D torus with major radius ``R`` and minor radius ``r``.

    Parametrised by ``u, v ~ Uniform[0, 2 pi)``:
    ``x = (R + r cos v) cos u``, ``y = (R + r cos v) sin u``, ``z = r sin v``.

    Returns:
        ``(n, 3)`` point cloud with one prominent ``H_2`` feature and one
        prominent ``H_1`` feature (the two independent cycles of the torus).
    """
    rng = _as_rng(rng)
    u = rng.uniform(0.0, 2.0 * np.pi, size=n)
    v = rng.uniform(0.0, 2.0 * np.pi, size=n)
    x = (R + r * np.cos(v)) * np.cos(u)
    y = (R + r * np.cos(v)) * np.sin(u)
    z = r * np.sin(v)
    pts = np.column_stack([x, y, z])
    return pts + rng.normal(scale=noise, size=pts.shape)


def sphere_cloud(n, radius=1.0, noise=0.05, rng=None):
    """Noisy 2-sphere: ``n`` points uniform on the sphere surface plus Gaussian jitter.

    Uses the standard ``z ~ Uniform(-1, 1)``, ``phi ~ Uniform(0, 2 pi)``
    parametrisation.

    Returns:
        ``(n, 3)`` point cloud with one prominent ``H_2`` feature of scale
        approximately ``radius``.
    """
    rng = _as_rng(rng)
    z = rng.uniform(-1.0, 1.0, size=n)
    phi = rng.uniform(0.0, 2.0 * np.pi, size=n)
    rho = np.sqrt(np.maximum(0.0, 1.0 - z**2))
    pts = radius * np.column_stack([rho * np.cos(phi), rho * np.sin(phi), z])
    return pts + rng.normal(scale=noise, size=pts.shape)


def cluster_cloud(n, n_clusters=3, spread=3.0, noise=0.2, rng=None):
    """Gaussian blob clusters: ``n`` points split across ``n_clusters`` blobs.

    Cluster centers sit on a lattice with spacing ``spread``; points within a
    cluster are Gaussian with standard deviation ``noise``.

    Returns:
        ``(n, 2)`` point cloud with ``n_clusters`` connected components at the
        ``H_0`` scale ``spread``.
    """
    rng = _as_rng(rng)
    if n_clusters < 1:
        raise ValueError("n_clusters must be >= 1")
    side = int(np.ceil(np.sqrt(n_clusters)))
    centers = []
    for k in range(n_clusters):
        i, j = k % side, k // side
        centers.append([(i - (side - 1) / 2) * spread, (j - (side - 1) / 2) * spread])
    centers = np.asarray(centers, dtype=float)
    sizes = np.full(n_clusters, n // n_clusters)
    sizes[: n % n_clusters] += 1
    pts = []
    for k, size in enumerate(sizes):
        pts.append(centers[k] + rng.normal(scale=noise, size=(size, 2)))
    return np.vstack(pts)


def split_cluster_cloud(n_per_blob, n_blobs, separation=3.0, noise=0.15,
                        deterministic=False, n_gon=12, rng=None):
    """Cloud of ``n_blobs`` Gaussian blobs on a regular polygon of side ``separation``.

    This is the WP1.1 / Phase 4.4 "cluster splitting" DGP: the H_0 diagram of a
    cloud with ``n_blobs`` well-separated blobs has ``n_blobs - 1`` finite
    classes, all dying at the blob-merge scale. Two blobs (one finite class)
    versus three blobs (two finite classes, equilateral arrangement so both
    classes share the merge law) give equal mean power-weighted silhouettes but
    different diagram laws; the mean is preserved because the silhouette is a
    normalized average and the merge-scale law is unchanged by the extra blob.

    With ``deterministic=True`` each blob is a fixed regular ``n_gon``-gon of
    radius ``noise`` (degenerate randomness), so under the radius-convention
    filtrations of ``tda2s.ph`` (alpha, Delaunay-Cech) the merge scale is
    exactly ``(separation - 2 * noise) / 2`` and the mean-silhouette equality
    holds realization by realization, not only in expectation. Requires
    ``n_gon`` divisible by 4 for ``n_blobs`` in {2, 3}, so that a vertex sits
    exactly on the inter-blob axis in both arrangements; with the default
    ``n_gon=12`` the within-blob classes all die at
    ``noise * sin(pi / n_gon)`` (0.0388 at the defaults), well below the
    persistence threshold used to isolate the merge classes.

    Note that the two arms differ in cardinality (``n_blobs * n_gon`` points),
    so any use of this generator as a null DGP must either state cloud size as
    part of the treatment or equalise it by subsampling.

    Args:
        n_per_blob: points per blob in the stochastic case (Gaussian around the
            blob centre with scale ``noise``). Ignored when
            ``deterministic=True``, which always emits ``n_gon`` vertices.
        n_blobs: number of blobs, placed at the vertices of a regular polygon
            with side length ``separation`` (2 blobs: a segment; 3 blobs: an
            equilateral triangle; more: the regular polygon, so consecutive
            neighbours are at distance ``separation``).
        separation: distance between neighbouring blob centres.
        noise: per-blob scale: Gaussian standard deviation (stochastic) or
            blob radius (deterministic).
        deterministic: use fixed regular polygons instead of Gaussian blobs.
        n_gon: vertices per blob in the deterministic case (default 12).
        rng: seed or Generator.

    Returns:
        ``(n, 2)`` point cloud.
    """
    rng = _as_rng(rng)
    n_blobs = int(n_blobs)
    if n_blobs < 2:
        raise ValueError("n_blobs must be >= 2 (one blob has no finite H_0 class)")
    # Vertices of a regular n-gon with consecutive side length `separation`.
    R = separation / (2.0 * np.sin(np.pi / n_blobs))
    angles = 2.0 * np.pi * np.arange(n_blobs) / n_blobs
    centers = R * np.column_stack([np.cos(angles), np.sin(angles)])

    pts = []
    for k in range(n_blobs):
        if deterministic:
            theta = 2.0 * np.pi * np.arange(int(n_gon)) / int(n_gon)
            blob = centers[k] + noise * np.column_stack([np.cos(theta), np.sin(theta)])
        else:
            blob = centers[k] + rng.normal(scale=noise, size=(n_per_blob, 2))
        pts.append(blob)
    return np.vstack(pts)


def loops_cloud(n, n_loops, radius=1.0, noise=0.05, outlier_fraction=0.0, rng=None):
    """Cloud of ``n_loops`` noisy circles arranged on a big circle.

    Each loop is a circle of its own radius; loop centers sit at equally spaced
    angles on a big circle whose radius guarantees the loops stay separated
    (adjacent centers at least 4 x the largest loop radius apart), so the alpha
    complex of the whole cloud has one prominent ``H_1`` feature per loop with
    persistence approximately equal to that loop's radius. A fraction
    ``outlier_fraction`` of the points are drawn uniformly over the bounding
    box of the arrangement (topological clutter).

    Args:
        n: total number of points.
        n_loops: number of loops (prominent ``H_1`` features).
        radius: loop radius; either a scalar (all loops equal) or a length
            ``n_loops`` array of per-loop radii.
        noise: standard deviation of the isotropic Gaussian jitter.
        outlier_fraction: fraction of points placed uniformly in the bounding
            box (default 0.0).
        rng: seed or Generator.

    Returns:
        ``(n, 2)`` point cloud.
    """
    rng = _as_rng(rng)
    n_loops = int(n_loops)
    if n_loops < 1:
        raise ValueError("n_loops must be >= 1")
    radii = np.full(n_loops, float(radius)) if np.isscalar(radius) else np.asarray(radius, dtype=float)
    if radii.shape[0] != n_loops:
        raise ValueError("radius array must have length n_loops")
    max_r = float(radii.max())
    n_outliers = int(round(outlier_fraction * n))
    n_loop_pts = n - n_outliers
    per_loop, remainder = divmod(n_loop_pts, n_loops)
    if per_loop < 8:
        raise ValueError("n too small for n_loops (need >= 8 points per loop)")

    big_R = 2.5 * max_r if n_loops > 1 else 0.0
    angles = 2.0 * np.pi * np.arange(n_loops) / n_loops
    centers = big_R * np.column_stack([np.cos(angles), np.sin(angles)])

    pts = []
    for k in range(n_loops):
        size = per_loop + (1 if k < remainder else 0)
        theta = rng.uniform(0.0, 2.0 * np.pi, size=size)
        circle = centers[k] + radii[k] * np.column_stack([np.cos(theta), np.sin(theta)])
        pts.append(circle + rng.normal(scale=noise, size=circle.shape))
    if n_outliers > 0:
        span = big_R + max_r
        pts.append(rng.uniform(-span, span, size=(n_outliers, 2)))
    return np.vstack(pts)

In [ ]:
%writefile tda2s/dgp/simulation.py
"""Covariate-driven point-cloud DGP harness (Phase 0.6).

The harness generates *datasets* of point clouds for two-sample topological
testing. It exposes independently dialable knobs:

  * covariates ``X ~ N(0, I)`` or a two-component Gaussian mixture;
  * propensity ``pi(X) = expit(prop_scale * X @ beta)`` (imbalance dialled via
    ``|beta|`` and ``prop_scale``);
  * covariate-driven topology: a deterministic function ``topology_knob(x) ->
    (n_loops, radius, noise)`` maps each covariate row to the generator
    parameters of its cloud;
  * an optional direct group effect that shifts the topology of group A
    regardless of ``X`` (power experiments).

Key design property: with ``group_effect=0`` the topology of a cloud is a
deterministic function of its covariate vector ``X_i``, so conditional on ``X``
the two groups have the *identical* topological law; only the propensity
differs between groups. This is the exact structure Phase 2's covariate-shift
gate experiments require. With ``group_effect > 0`` the group-A loop count is
shifted regardless of ``X``.

The harness records, per cloud, the true generator parameters (``n_loops``,
per-loop ``radii``, ``noise``, ``outlier_fraction``) in ``CloudSample.oracle``,
so tests and experiments can verify that dialled knobs are recovered from the
oracle.

``to_silhouette_sample`` converts a sample of clouds to the observed triplet
``(phi, A, X)`` in the ``tcda_uq`` silhouette convention (``phi`` of shape
``[n, n_hom_dim, resolution]``), so Phase 3 can feed both harnesses through the
same downstream estimators.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy.special import expit

from .clouds import loops_cloud

_MU1 = np.array([1.0, 0.6, -0.7, 2.2, -1.0])
_MU2 = np.array([0.4, -0.4, -0.6, 3.3, 3.0])
_BETA = np.array([-0.5, -0.1, 0.6, 0.1, 0.1])


@dataclass
class CloudSample:
    """One realised dataset of point clouds with per-cloud topology oracles.

    Attributes:
        clouds: list of ``(m_i, 2)`` point clouds, one per unit.
        X: ``[n, d_x]`` covariate matrix.
        A: ``[n]`` group labels (0 = B, 1 = A).
        propensity: ``[n]`` true propensity ``pi(X)``.
        oracle: dict mapping unit index to ``{"n_loops", "radii", "noise",
            "outlier_fraction"}``, the exact generator parameters used.
    """

    clouds: list
    X: np.ndarray
    A: np.ndarray
    propensity: np.ndarray
    oracle: dict

    @property
    def true_n_loops(self) -> np.ndarray:
        """``[n]`` oracle loop counts."""
        return np.array([self.oracle[i]["n_loops"] for i in range(len(self.clouds))])

    @property
    def true_radii(self) -> list:
        """Per-cloud oracle loop radii (list of per-loop arrays)."""
        return [self.oracle[i]["radii"] for i in range(len(self.clouds))]

    @property
    def true_noise(self) -> np.ndarray:
        """``[n]`` oracle point jitter scales."""
        return np.array([self.oracle[i]["noise"] for i in range(len(self.clouds))])

    def observed(self, **sil_kwargs):
        """Observed triplet ``(phi, A, X)`` in tcda_uq silhouette format."""
        return to_silhouette_sample(self.clouds, self.X, self.A, **sil_kwargs)


def _default_topology_knob(gamma, k_max, radius, noise):
    """Default knob: ``n_loops = 1 + floor(expit(gamma * x0) * k_max)``."""

    def knob(x):
        n_loops = 1 + int(np.floor(expit(gamma * x[0]) * k_max))
        return n_loops, radius, noise

    return knob


class CloudSampleDGP:
    """Two-group point-cloud DGP with covariate-driven topology.

    Args:
        n_per_group: half the dataset size; ``n = 2 * n_per_group`` clouds are
            drawn in total. Labels are then drawn as ``A_i ~ Bern(pi(X_i))``, so
            the two groups are *not* forced to be equal-sized -- that imbalance
            is exactly the confounding this harness exists to create. Pass
            ``beta=np.zeros(d_x)`` for a balanced, unconfounded design.
        m: number of points per cloud.
        d_x: covariate dimension.
        covariate: ``"gaussian"`` (standard normal) or ``"mixture"``
            (two-component Gaussian mixture, tcda_uq conventions).
        beta: propensity coefficients (``None`` uses the tcda_uq default).
        prop_scale: multiplier of the propensity logit (``> 1`` pushes
            ``pi(X)`` toward ``{0, 1}``).
        topology_knob: callable ``x -> (n_loops, radius, noise)`` mapping one
            covariate row to generator parameters. ``None`` uses
            ``n_loops = 1 + floor(expit(gamma * x[0]) * k_max)`` with fixed
            ``radius`` and ``noise``. A deterministic knob is what makes the
            two groups conditionally identical given ``X``.
        gamma, k_max: slope and max extra loops of the default knob.
        radius, noise, outlier_fraction: fixed generator parameters used when
            ``topology_knob`` is ``None``.
        group_effect: integer loop-count shift applied to group A regardless
            of ``X`` (0 = conditionally identical groups).
        seed: recorded on the instance for provenance; the model coefficients
            (mixture means, default ``beta``) are fixed constants, so sampling
            randomness is controlled entirely by ``sample(rng=...)``.
    """

    def __init__(
        self,
        n_per_group: int = 25,
        m: int = 120,
        d_x: int = 3,
        covariate: str = "gaussian",
        beta=None,
        prop_scale: float = 1.0,
        topology_knob=None,
        gamma: float = 1.0,
        k_max: int = 3,
        radius: float = 1.0,
        noise: float = 0.05,
        outlier_fraction: float = 0.0,
        group_effect: int = 0,
        seed: int = 0,
    ):
        if n_per_group < 1:
            raise ValueError("n_per_group must be >= 1")
        if m < 40:
            raise ValueError("m must be >= 40 to keep loop persistence recoverable")
        if covariate not in ("gaussian", "mixture"):
            raise ValueError("covariate must be 'gaussian' or 'mixture'")

        self.n_per_group = int(n_per_group)
        self.m = int(m)
        self.d_x = int(d_x)
        self.covariate = covariate
        self.prop_scale = float(prop_scale)
        self.gamma = float(gamma)
        self.k_max = int(k_max)
        self.radius = float(radius)
        self.noise = float(noise)
        self.outlier_fraction = float(outlier_fraction)
        self.group_effect = int(group_effect)

        self.seed = seed
        self.mu1 = _MU1[: self.d_x]
        self.mu2 = _MU2[: self.d_x]
        self.Sigma = np.eye(self.d_x) * 0.5
        self.beta = np.asarray(beta if beta is not None else _BETA[: self.d_x], dtype=float)
        if self.beta.shape[0] != self.d_x:
            raise ValueError("beta must have length d_x")

        self.topology_knob = topology_knob if topology_knob is not None else _default_topology_knob(
            self.gamma, self.k_max, self.radius, self.noise
        )
        self._max_loops = max(1, self.m // 8)

    def propensity(self, X):
        """True propensity ``pi(X) = expit(prop_scale * X @ beta)``, ``[n]``."""
        return expit(self.prop_scale * (np.asarray(X, dtype=float) @ self.beta))

    def topology(self, x):
        """Topology tuple ``(n_loops, radius, noise)`` for one covariate row."""
        return self.topology_knob(np.asarray(x, dtype=float))

    def _sample_covariates(self, n, rng):
        if self.covariate == "gaussian":
            return rng.normal(size=(n, self.d_x))
        n1 = n // 2
        X1 = rng.multivariate_normal(self.mu1, self.Sigma, size=n1)
        X2 = rng.multivariate_normal(self.mu2, self.Sigma, size=n - n1)
        return np.vstack([X1, X2])

    def sample(self, n_per_group=None, X=None, rng=None) -> CloudSample:
        """Draw a dataset of ``2 * n_per_group`` point clouds.

        Args:
            n_per_group: overrides the constructor default.
            X: optional fixed covariate matrix ``[n, d_x]`` (e.g. repeated rows
                for conditional-identical-group checks); drawn otherwise.
            rng: seed or Generator.

        Returns:
            :class:`CloudSample` with ``clouds``, ``X``, ``A``, ``propensity``
            and the per-cloud ``oracle`` of true generator parameters.
        """
        rng = rng if isinstance(rng, np.random.Generator) else np.random.default_rng(rng)
        n_per_group = self.n_per_group if n_per_group is None else int(n_per_group)
        n = 2 * n_per_group
        if X is None:
            X = self._sample_covariates(n, rng)
        else:
            X = np.asarray(X, dtype=float)
            if X.shape != (n, self.d_x):
                raise ValueError(f"X must have shape ({n}, {self.d_x})")

        pi = self.propensity(X)
        A = rng.binomial(1, pi).astype(int)

        clouds, oracle = [], {}
        for i in range(n):
            n_loops, radius, noise = self.topology(X[i])
            if A[i] == 1:
                n_loops = int(n_loops) + self.group_effect
            n_loops = int(np.clip(n_loops, 1, self._max_loops))
            radii = np.full(n_loops, float(radius))
            cloud = loops_cloud(self.m, n_loops, radius=radii, noise=noise,
                                outlier_fraction=self.outlier_fraction, rng=rng)
            clouds.append(cloud)
            oracle[i] = {
                "n_loops": n_loops,
                "radii": radii.copy(),
                "noise": float(noise),
                "outlier_fraction": self.outlier_fraction,
            }
        return CloudSample(clouds=clouds, X=X, A=A, propensity=pi, oracle=oracle)


# --- Phase 2.2 masking DGP (exact Simpson cancellation) -----------------------
# Three covariate strata X in {0, 1, 2} (uniform), propensity e = (1/2, 1/2, 1/4).
# Conditional diagram laws: strata 0, 1 have identical laws in both arms; the
# treatment effect lives entirely in stratum 2, where the treated law L_C and
# the control law (4/15)(L_A + L_B) + (7/15)L_C are chosen so that the marginal
# observational laws coincide exactly:
#
#     L(D|A=1) = (2/5)(L_A + L_B) + (1/5)L_C  =  L(D|A=0).
#
# This is exact (verified in tests/test_phase2.py): every valid level-alpha
# permutation test of H0^cond has power exactly alpha there, while the
# covariate-standardized topological effect is
#
#     psi_d = (1/3)(8/15)(m_C - (m_A + m_B) / 2) != 0
#
# whenever the mean silhouettes of the three types do not satisfy m_C =
# (m_A + m_B) / 2 (here m_A = Lambda_{r_A}, m_B = Lambda_{r_B}, m_C = Lambda_{r_C}
# on the persistence scale ~0.75 * radius, with r_A = 1, r_B = 2, r_C = 4).

_MASK_E = np.array([0.5, 0.5, 0.25])          # per-stratum propensity
_MASK_W0 = 4.0 / 15.0                          # control-mixture weights at stratum 2
_MASK_W1 = 7.0 / 15.0


def masking_stratum_sample(n_per_group, m=120, noise=0.05, radius_a=1.0,
                           radius_b=2.0, radius_c=4.0, seed=None):
    """Exact Simpson-masking DGP: H0^cond true, H0^out false (Phase 2.2).

    Stratum 0 and 1 units are 1-loop clouds of radius ``radius_a`` / ``radius_b``
    in *both* arms (no treatment effect there). Stratum 2 treated units are
    2-loop clouds of radius ``radius_c``; stratum 2 control units draw a cloud
    type from the mixture ``(4/15, 4/15, 7/15)`` over the three types. With
    propensity e = (1/2, 1/2, 1/4) the marginal observational diagram laws
    coincide exactly between arms, while the covariate-standardized silhouette
    effect is non-zero.

    Args:
        n_per_group: number of units per arm of the sample.
        m: points per cloud.
        noise: per-cloud Gaussian jitter scale.
        radius_a, radius_b, radius_c: loop radii of the three cloud types.
        seed: RNG seed.

    Returns:
        :class:`CloudSample` with ``X`` in {0, 1, 2}, the true propensity, and
        an oracle recording per-unit (stratum, cloud type, radii).
    """
    rng = np.random.default_rng(seed)
    n = 2 * int(n_per_group)
    X = np.tile(np.arange(3), int(np.ceil(n / 3)))[:n]
    e = _MASK_E[X]
    A = rng.binomial(1, e).astype(int)

    def _cloud(radius, n_loops):
        return loops_cloud(m, n_loops, radius=radius, noise=noise, rng=rng)

    clouds, oracle = [], {}
    for i in range(n):
        x, a = int(X[i]), int(A[i])
        if x in (0, 1):
            radius = radius_a if x == 0 else radius_b
            n_loops = 1
        else:  # stratum 2: the only stratum carrying the effect
            if a == 1:
                radius, n_loops = radius_c, 2
            else:
                u = rng.random()
                if u < _MASK_W0:
                    radius, n_loops = radius_a, 1
                elif u < 2 * _MASK_W0:
                    radius, n_loops = radius_b, 1
                else:
                    radius, n_loops = radius_c, 2
        clouds.append(_cloud(radius, n_loops))
        oracle[i] = {"stratum": x, "n_loops": n_loops,
                     "radii": np.full(n_loops, float(radius)), "noise": noise}
    return CloudSample(clouds=clouds, X=np.asarray(X, dtype=float).reshape(-1, 1),
                       A=A, propensity=e, oracle=oracle)


def _silhouette_from_diagrams(diags, interval, r, resolution):
    """Power-weighted silhouette of a diagram list (tcda_uq convention).

    Delegates to ``tda2s.vec.silhouette``, which uses the same power-weight
    convention (``|death - birth| ** r``, ``keep_endpoints=True``) as
    ``tcda_uq.silhouette.compute_silhouette``.
    """
    from tda2s.vec import silhouette

    return silhouette(diags, interval=interval, r=r, resolution=resolution)


def to_silhouette_sample(clouds, X, A, filtration="alpha", homology_dims=(0, 1),
                         interval=(0.0, 1.0), r=3.0, resolution=100, **ph_kwargs):
    """Convert clouds to the observed ``(phi, A, X)`` silhouette triplet.

    Each cloud is mapped to persistence diagrams via
    ``tda2s.ph.compute_diagrams`` and then to a power-weighted silhouette, so
    the output matches ``tcda_uq``'s observed format: ``phi`` has shape
    ``[n, n_hom_dim, resolution]``, ``A`` is ``[n]`` and ``X`` is ``[n, d_x]``.

    Args:
        clouds: iterable of ``(m_i, 2)`` point clouds.
        X: ``[n, d_x]`` covariate matrix.
        A: ``[n]`` group labels.
        filtration, homology_dims: passed to ``tda2s.ph.compute_diagrams``.
        interval, r, resolution: silhouette domain, power-weight exponent and
            grid size (tcda_uq conventions).
        **ph_kwargs: extra ``compute_diagrams`` keyword arguments.

    Returns:
        Tuple ``(phi, A, X)`` with ``phi`` of shape ``[n, n_hom_dim, resolution]``.
    """
    from tda2s.ph import compute_diagrams

    n_hom = len(homology_dims)
    n = len(clouds)
    phi = np.empty((n, n_hom, resolution))
    for i, cloud in enumerate(clouds):
        diags = compute_diagrams(cloud, filtration=filtration,
                                 homology_dims=homology_dims, **ph_kwargs)
        phi[i] = _silhouette_from_diagrams(diags, interval=interval, r=r,
                                           resolution=resolution)
    return phi, np.asarray(A, dtype=int), np.asarray(X, dtype=float)

In [ ]:
%writefile tda2s/ph/__init__.py
"""PH pipeline: point cloud -> persistence diagrams, uniform API.

Filtrations: VR (gudhi), ripser (fast), Alpha, Cech, cubical-sublevel (grid
distance transform), DTM-Rips (weighted Rips with DTM vertex weights).

Conventions
-----------
* A diagram is a ``(k, 2)`` float array of (birth, death) pairs, one per
  homology dimension, returned as a list indexed by homology dim.
* Essential classes (death = inf) are dropped: with filtrations of compact
  point sets every class dies, and tcda_uq uses the same convention.
* All filtration values are radii. Alpha *and* Delaunay-Cech report squared
  circumradii internally, so both extractors take a square root; Rips/ripser
  and DTM-Rips are already on the radius scale.
* Diagrams are cached to disk keyed by (point-cloud hash, filtration, params):
  permutation tests must never recompute PH inside the permutation loop.
* gudhi pair format: ``st.persistence()`` yields ``(dim, (birth, death))``
  tuples with essential classes as ``(dim, (birth, inf))``. Extractors are
  defensive about non-tuple payloads anyway (see ``_drop_infinite``), and
  infinite deaths are always dropped, so downstream code never sees infs.
* ``dtm-rips`` with ``homology_dims`` including 2 and an unbounded
  ``max_edge_length`` enumerates every 3-simplex of the cloud (combinatorial
  blow-up); pass a bounded ``max_edge_length`` for that configuration.
"""
from __future__ import annotations

import hashlib
import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

_HOMOLOGY_DIMS = (0, 1, 2)


@dataclass
class PhParams:
    """Filtration parameters (cached under the hash of these + the cloud)."""

    filtration: str = "alpha"
    homology_dims: Tuple[int, ...] = _HOMOLOGY_DIMS
    max_edge_length: Optional[float] = None
    grid_size: int = 64
    dtm_k: int = 20
    cache_dir: Optional[str] = None

    def key(self, points: np.ndarray) -> str:
        h = hashlib.sha256()
        h.update(np.ascontiguousarray(points, dtype=np.float32).tobytes())
        h.update(repr((self.filtration, self.homology_dims, self.max_edge_length,
                       self.grid_size, self.dtm_k)).encode())
        return h.hexdigest()[:24]


def _points_to_float(points) -> np.ndarray:
    pts = np.asarray(points, dtype=float)
    if pts.ndim != 2:
        raise ValueError(f"points must be (m, d), got {pts.shape}")
    return pts


def _drop_infinite(dgm: np.ndarray) -> np.ndarray:
    dgm = np.asarray(dgm, dtype=float)
    if dgm.ndim != 2:
        return dgm.reshape(0, 2)
    if dgm.size == 0:
        return dgm.reshape(0, 2)
    return dgm[np.isfinite(dgm[:, 1])]


def _alpha_diagrams(pts: np.ndarray, max_dim: int, max_edge_length) -> List[np.ndarray]:
    import gudhi as gd

    alpha = gd.AlphaComplex(points=pts)
    st = alpha.create_simplex_tree()
    if max_edge_length is not None:
        st.prune_above_filtration(max_edge_length**2)
    st.compute_persistence()
    out = []
    for d in range(max_dim + 1):
        finite = [p for p in st.persistence() if p[0] == d
                  and isinstance(p[1], tuple) and np.isfinite(p[1][1])]
        dgm = np.array([p[1] for p in finite], dtype=float).reshape(-1, 2)
        out.append(np.sqrt(dgm) if dgm.size else dgm)
    return out


def _vr_diagrams(pts: np.ndarray, max_dim: int, max_edge_length) -> List[np.ndarray]:
    import gudhi as gd

    rc = gd.RipsComplex(points=pts, max_edge_length=max_edge_length or np.inf)
    st = rc.create_simplex_tree(max_dimension=max_dim + 1)
    st.compute_persistence()
    return [_drop_infinite(np.array(
        [p[1] for p in st.persistence() if p[0] == d], dtype=float).reshape(-1, 2))
        for d in range(max_dim + 1)]


def _ripser_diagrams(pts: np.ndarray, max_dim: int, max_edge_length) -> List[np.ndarray]:
    import ripser

    dgms = ripser.ripser(pts, maxdim=max_dim, thresh=float(max_edge_length) if max_edge_length else np.inf)["dgms"]
    out = []
    for d in range(max_dim + 1):
        dgm = _drop_infinite(np.asarray(dgms[d], dtype=float))
        out.append(dgm)
    return out


def _cech_diagrams(pts: np.ndarray, max_dim: int, max_edge_length) -> List[np.ndarray]:
    import gudhi as gd

    cc = gd.DelaunayCechComplex(points=pts)
    st = cc.create_simplex_tree()
    if max_edge_length is not None:
        st.prune_above_filtration(max_edge_length**2)
    st.compute_persistence()
    out = []
    for d in range(max_dim + 1):
        dgm = _drop_infinite(np.array(
            [p[1] for p in st.persistence() if p[0] == d], dtype=float).reshape(-1, 2))
        # DelaunayCechComplex reports *squared* circumradii (same convention as
        # AlphaComplex); take the square root so every filtration in this module
        # is on the radius scale.
        out.append(np.sqrt(dgm) if dgm.size else dgm)
    return out


def _cubical_diagrams(pts: np.ndarray, max_dim: int, grid_size: int) -> List[np.ndarray]:
    """Cubical sublevel filtration of the distance-to-cloud function on a grid.

    The grid distance transform is a piecewise-Lipschitz proxy for the
    distance function; its sublevel sets reproduce the cloud's topology at
    scales above the grid resolution.
    """
    import gudhi as gd
    from scipy.spatial import cKDTree

    # Pad relative to the cloud's extent, not by an absolute epsilon: an
    # absolute pad would make the grid (and hence the filtration) depend on the
    # cloud's units, breaking scale equivariance at the 1e-6 level.
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    pad = 1e-6 * float(np.max(hi - lo))
    pad = pad if pad > 0 else 1e-6
    lo, hi = lo - pad, hi + pad
    axes = [np.linspace(lo[j], hi[j], grid_size) for j in range(pts.shape[1])]
    grid = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1).reshape(-1, pts.shape[1])
    dist = cKDTree(pts).query(grid, k=1)[0].reshape([grid_size] * pts.shape[1])
    cc = gd.CubicalComplex(dimensions=list(dist.shape), top_dimensional_cells=dist.ravel())
    cc.compute_persistence()
    return [_drop_infinite(np.array(cc.persistence_intervals_in_dimension(d), dtype=float).reshape(-1, 2))
            for d in range(min(max_dim, 2) + 1)]


def _dtm_rips_diagrams(pts: np.ndarray, max_dim: int, dtm_k: int, max_edge_length) -> List[np.ndarray]:
    """Weighted Rips with DTM vertex weights (Anai et al. 2019 construction)."""
    import numpy as np
    from scipy.spatial.distance import cdist
    from gudhi.point_cloud.dtm import DistanceToMeasure
    from gudhi.weighted_rips_complex import WeightedRipsComplex

    # DistanceToMeasure is not callable in gudhi 3.11 (dtm(pts) raises
    # TypeError); transform after fit.
    dtm = DistanceToMeasure(k=dtm_k)
    weights = np.asarray(dtm.fit_transform(pts), dtype=float)
    # gudhi's WeightedRipsComplex expects a pairwise distance matrix.
    dist = cdist(pts, pts)
    rc = WeightedRipsComplex(distance_matrix=dist, weights=weights,
                             max_filtration=max_edge_length if max_edge_length is not None else np.inf)
    st = rc.create_simplex_tree(max_dimension=max_dim + 1)
    st.compute_persistence()
    return [_drop_infinite(np.array(
        [p[1] for p in st.persistence() if p[0] == d], dtype=float).reshape(-1, 2))
        for d in range(max_dim + 1)]


def compute_diagrams(points, filtration: str = "alpha",
                     homology_dims: Sequence[int] = _HOMOLOGY_DIMS,
                     max_edge_length: Optional[float] = None,
                     grid_size: int = 64, dtm_k: int = 20,
                     standardise: Optional[Tuple[np.ndarray, np.ndarray]] = None,
                     cache_dir: Optional[str] = None) -> List[np.ndarray]:
    """Compute persistence diagrams of a point cloud under ``filtration``.

    Args:
        points: ``(m, d)`` point cloud.
        filtration: one of {"vr", "ripser", "alpha", "cech", "cubical", "dtm-rips"}.
        homology_dims: homology dimensions to keep.
        max_edge_length: filtration cutoff (radius units; None = unbounded).
        grid_size: grid edge length for "cubical".
        dtm_k: DTM neighbourhood size for "dtm-rips".
        standardise: optional ``(mean, scale)`` pair of ``(d,)`` arrays applied
            to the points as ``(points - mean) / scale`` before filtration.
            Externally supplied by the caller (e.g. a fixed study-region
            background) instead of per-cloud scaling, so diagrams of different
            clouds live in one shared coordinate system.
        cache_dir: if set, diagrams are cached/loaded keyed by cloud+params hash.

    Returns:
        List of ``(k, 2)`` (birth, death) arrays, indexed by homology dim.
    """
    pts = _points_to_float(points)
    if standardise is not None:
        mean, scale = standardise
        pts = (pts - np.asarray(mean, dtype=float)) / np.asarray(scale, dtype=float)
    dims = tuple(int(d) for d in homology_dims)
    max_dim = max(dims)
    params = PhParams(filtration=filtration, homology_dims=dims,
                      max_edge_length=max_edge_length, grid_size=grid_size,
                      dtm_k=dtm_k, cache_dir=cache_dir)
    if cache_dir:
        os.makedirs(cache_dir, exist_ok=True)
        path = os.path.join(cache_dir, f"{params.key(pts)}.npz")
        if os.path.exists(path):
            with np.load(path, allow_pickle=False) as z:
                return [z[f"d{d}"] for d in dims]

    if filtration == "alpha":
        all_d = _alpha_diagrams(pts, max_dim, max_edge_length)
    elif filtration == "vr":
        all_d = _vr_diagrams(pts, max_dim, max_edge_length)
    elif filtration == "ripser":
        all_d = _ripser_diagrams(pts, max_dim, max_edge_length)
    elif filtration == "cech":
        all_d = _cech_diagrams(pts, max_dim, max_edge_length)
    elif filtration == "cubical":
        all_d = _cubical_diagrams(pts, max_dim, grid_size)
    elif filtration == "dtm-rips":
        all_d = _dtm_rips_diagrams(pts, max_dim, dtm_k, max_edge_length)
    else:
        raise ValueError(f"unknown filtration: {filtration}")

    out = [all_d[d] for d in dims]
    if cache_dir:
        np.savez(path, **{f"d{d}": arr for d, arr in zip(dims, out)})
    return out


def betti_numbers(diags: Sequence[np.ndarray], persistence_threshold: float) -> List[int]:
    """Count features with persistence strictly above a threshold, per dim.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays (see ``compute_diagrams``).
        persistence_threshold: features with ``death - birth > threshold`` count.

    Returns:
        One count per homology dim.
    """
    counts = []
    for dgm in diags:
        dgm = np.asarray(dgm, dtype=float).reshape(-1, 2)
        counts.append(int((dgm[:, 1] - dgm[:, 0] > persistence_threshold).sum()))
    return counts

In [ ]:
%writefile tda2s/vec/__init__.py
"""Vectorisation stack: persistence diagrams -> fixed-size feature vectors.

Uniform entry point ``vectorise(diags, representation, **kwargs)`` dispatches
to per-representation functions. Representations:

* ``silhouette``: power-weighted persistence silhouette (gudhi
  ``representations.Silhouette``), same parameterisation as
  ``tcda_uq.silhouette.core.compute_silhouette`` (interval (0.0, 0.2),
  resolution 100, keep_endpoints=True, power r=3).
* ``landscape``: persistence landscapes (gudhi ``representations.Landscape``).
* ``betti``: Betti curves ``B_d(t) = #{features: birth <= t < death}`` on a
  t-grid (implemented locally).
* ``euler``: Euler curves ``sum_d (-1)^d B_d(t)``.
* ``image``: persistence images (gudhi ``representations.PersistenceImage``).
* ``measure``: persistence measure (Divol-Lacombe style): the diagram is a
  measure ``mu = sum_p w_p * delta_{(b, (b+d)/2)}`` over the (birth, mid)
  plane, projected onto a fixed 2-D grid of bins.

Conventions
-----------
* ``diags``: list of ``(k, 2)`` (birth, death) arrays, one per homology dim
  (same format as ``tda2s.ph.compute_diagrams``); deaths must be finite
  (essential classes are dropped upstream).
* Each homology dim is vectorised separately, so outputs carry a leading
  per-dim axis; all outputs are float64 arrays.
* ``interval`` (or ``sample_range``) defaults, when not given, to ``(0.0,
  max death)`` derived from the diagrams.
"""
from __future__ import annotations

from typing import Callable, List, Optional, Sequence, Tuple

import numpy as np


def _diagram_list(diags: Sequence[np.ndarray]) -> List[np.ndarray]:
    """Coerce the per-dim diagram list to a list of (k, 2) float arrays."""
    out = []
    for dgm in diags:
        out.append(np.asarray(dgm, dtype=float).reshape(-1, 2))
    return out


def _default_interval(diags: Sequence[np.ndarray]) -> Tuple[float, float]:
    """Default sample range ``(0, max death)`` across all dims."""
    hi = 1.0
    for dgm in _diagram_list(diags):
        if len(dgm):
            hi = max(hi, float(dgm[:, 1].max()))
    return (0.0, hi)


def _power_weight(point: np.ndarray, r: float) -> float:
    """Power weight ``|death - birth|**r`` of a persistence point."""
    return float(np.abs(point[1] - point[0]) ** r)


def silhouette(diags: Sequence[np.ndarray], interval=(0.0, 0.2), r: float = 3.0,
               resolution: int = 100) -> np.ndarray:
    """Power-weighted persistence silhouette.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        interval: sample range ``[t_min, t_max]`` of the silhouette.
        r: power-weight exponent ``w = (death - birth)**r``.
        resolution: number of grid points.

    Returns:
        ``(n_dims, resolution)`` array of silhouette values.
    """
    from gudhi.representations import Silhouette

    s = Silhouette(weight=lambda x: _power_weight(x, r), resolution=resolution,
                   sample_range=list(interval), keep_endpoints=True)
    return np.asarray(s.fit_transform(_diagram_list(diags)), dtype=float)


def landscape(diags: Sequence[np.ndarray], num_landscapes: int = 5,
              resolution: int = 100, interval: Optional[Tuple[float, float]] = None) -> np.ndarray:
    """Persistence landscapes of each dim's diagram.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        num_landscapes: number of landscape functions per dim.
        resolution: number of grid points per landscape.
        interval: sample range; defaults to ``(0, max death)``.

    Returns:
        ``(n_dims, num_landscapes, resolution)`` array of landscape values.
    """
    from gudhi.representations import Landscape

    iv = interval if interval is not None else _default_interval(diags)
    l = Landscape(num_landscapes=num_landscapes, resolution=resolution,
                  sample_range=list(iv))
    out = np.asarray(l.fit_transform(_diagram_list(diags)), dtype=float)
    return out.reshape(len(diags), num_landscapes, resolution)


def betti_curve(diags: Sequence[np.ndarray], interval: Optional[Tuple[float, float]] = None,
                n_points: int = 100) -> np.ndarray:
    """Betti curves ``B_d(t) = #{features: birth <= t < death}``.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        interval: t-grid range; defaults to ``(0, max death)``.
        n_points: number of t-grid points.

    Returns:
        ``(n_dims, n_points)`` array of Betti numbers on the t-grid.
    """
    iv = interval if interval is not None else _default_interval(diags)
    grid = np.linspace(iv[0], iv[1], n_points)
    rows = []
    for dgm in _diagram_list(diags):
        if len(dgm) == 0:
            rows.append(np.zeros(n_points))
            continue
        alive = (dgm[:, 0][:, None] <= grid[None, :]) & (grid[None, :] < dgm[:, 1][:, None])
        rows.append(alive.sum(axis=0, dtype=np.int64).astype(float))
    return np.stack(rows)


def euler_curve(diags: Sequence[np.ndarray], interval: Optional[Tuple[float, float]] = None,
                n_points: int = 100) -> np.ndarray:
    """Euler characteristic curve ``sum_d (-1)^d B_d(t)``.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        interval: t-grid range; defaults to ``(0, max death)``.
        n_points: number of t-grid points.

    Returns:
        ``(n_points,)`` array of Euler characteristics on the t-grid.
    """
    b = betti_curve(diags, interval=interval, n_points=n_points)
    signs = np.array([(-1.0) ** d for d in range(len(diags))])
    return signs @ b


def persistence_image(diags: Sequence[np.ndarray], bandwidth: float = 0.1,
                      weight: Optional[Callable[[np.ndarray], float]] = None,
                      resolution=(10, 10),
                      interval: Optional[Tuple[float, float]] = None) -> np.ndarray:
    """Persistence images (gaussian kernels centred on diagram points).

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        bandwidth: gaussian kernel width.
        weight: point weight function; defaults to persistence ``death - birth``.
        resolution: ``(n_pixels_x, n_pixels_y)`` grid.
        interval: ``[t_min, t_max]`` shared by both axes; defaults to
            ``(0, max death)``.

    Returns:
        ``(n_dims, n_pixels_x, n_pixels_y)`` array of image intensities.
    """
    from gudhi.representations import PersistenceImage

    iv = interval if interval is not None else _default_interval(diags)
    w = weight if weight is not None else lambda x: x[1] - x[0]
    pi = PersistenceImage(bandwidth=bandwidth, weight=w,
                          resolution=list(resolution),
                          im_range=[iv[0], iv[1], iv[0], iv[1]])
    out = np.asarray(pi.fit_transform(_diagram_list(diags)), dtype=float)
    return out.reshape(len(diags), int(resolution[0]), int(resolution[1]))


def persistence_measure(diags: Sequence[np.ndarray],
                        weight: Optional[Callable[[np.ndarray], float]] = None,
                        interval: Optional[Tuple[float, float]] = None,
                        n_bins: int = 32) -> np.ndarray:
    """Persistence measure: ``mu = sum_p w_p * delta_{(b, (b+d)/2)}``.

    The diagram is represented as a weighted point measure on the (birth,
    mid) plane with ``mid = (b + d) / 2`` (Divol-Lacombe coordinates),
    projected onto a fixed regular grid of bins as a weighted 2-D histogram.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        weight: point weight function; defaults to persistence ``death - birth``.
        interval: shared ``[t_min, t_max]`` for both the birth and mid axes;
            defaults to ``(0, max death)``.
        n_bins: number of bins along each axis.

    Returns:
        ``(n_dims, n_bins, n_bins)`` array of aggregated weights per bin.
    """
    iv = interval if interval is not None else _default_interval(diags)
    w = weight if weight is not None else lambda x: x[1] - x[0]
    edges = np.linspace(iv[0], iv[1], n_bins + 1)
    out = []
    for dgm in _diagram_list(diags):
        if len(dgm) == 0:
            out.append(np.zeros((n_bins, n_bins)))
            continue
        mid = (dgm[:, 0] + dgm[:, 1]) / 2.0
        ws = np.array([w(p) for p in dgm], dtype=float)
        hist, _, _ = np.histogram2d(dgm[:, 0], mid, bins=[edges, edges], weights=ws)
        out.append(hist)
    return np.stack(out)


def vectorise(diags: Sequence[np.ndarray], representation: str, **kwargs) -> np.ndarray:
    """Vectorise persistence diagrams under a named representation.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        representation: one of {"silhouette", "landscape", "betti", "euler",
            "image", "measure"}.
        **kwargs: passed to the per-representation function (e.g. ``interval``,
            ``resolution``, ``r``).

    Returns:
        The representation vector; see the individual functions for shapes.
    """
    if representation == "silhouette":
        return silhouette(diags, **kwargs)
    if representation == "landscape":
        return landscape(diags, **kwargs)
    if representation == "betti":
        return betti_curve(diags, **kwargs)
    if representation == "euler":
        return euler_curve(diags, **kwargs)
    if representation == "image":
        return persistence_image(diags, **kwargs)
    if representation == "measure":
        return persistence_measure(diags, **kwargs)
    raise ValueError(
        f"unknown representation {representation!r}; expected one of "
        "['silhouette', 'landscape', 'betti', 'euler', 'image', 'measure']")


In [ ]:
%writefile experiments/phase3_dr_calibration.py
"""Phase 3 experiments for the outcome-level doubly robust test.

The default ``oracle`` design is the controlled functional benchmark from
``tcda_uq.datasets.TriOracleSimulation``.  It is intentionally fast enough to
produce the required size/power tables at n in {50, 100, 200, 500}.  The
``clouds`` design runs the same cached testing layer on the project's
covariate-driven point-cloud DGP, but is substantially slower because
persistent homology is computed once per replication.

Long runs are sharded by replication index:

    python experiments/phase3_dr_calibration.py --mode shard \
        --shard-idx 0 --reps-per-shard 10

The shard JSON files are independent checkpoints.  After the fleet is
complete, aggregate them with ``--mode aggregate``.  No permutation draw
recomputes persistent homology, cross-fitting, or a nuisance regression.
"""

from __future__ import annotations

import argparse
import json
import os
from collections import defaultdict

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from tda2s.dgp import CloudSampleDGP, to_silhouette_sample
from tda2s.tests.dr_outcome import (
    equivalence_test,
    fit_dr,
    multiplier_test,
    positivity_diagnostics,
    propensity_learner_grid,
    propensity_strata,
    stratified_permutation_test,
)

BASE_SEED = 3100
ALPHA = 0.05
SAMPLE_SIZES = (50, 100, 200, 500)
REGIMES = (0.0, 0.5, 1.0)
N_BASIS = 5
N_FOLDS = 2
N_CALIBRATION = 399
INTERVAL = (0.0, 1.0)
RESOLUTION = 50
_HERE = os.path.dirname(os.path.abspath(__file__))
RESULTS = os.path.join(_HERE, "..", "results")
SHARDS = os.path.join(RESULTS, "phase3_shards")


def _seed(*parts) -> int:
    value = BASE_SEED
    for part in parts:
        if isinstance(part, str):
            part = sum((i + 1) * byte for i, byte in enumerate(part.encode()))
        value = (value * 7919 + int(part)) % (2 ** 31 - 1)
    return int(value)


def _interaction_features(X):
    """Features matching TriOracleSimulation's logistic propensity law."""
    X = np.asarray(X)
    return np.column_stack([X, X[:, 1] * X[:, 2], X[:, 0] * X[:, 2]])


def _tri_sample(n: int, rep: int, regime: float, *, alternative: bool,
                true_n_basis: int = N_BASIS):
    """Draw a tri-oracle sample, optionally imposing the sharp outcome null."""
    sim = __import__("tcda_uq.datasets", fromlist=["TriOracleSimulation"]).TriOracleSimulation(
        n_cov=3, n_hom_dim=2, resolution=RESOLUTION, interval=INTERVAL,
        n_basis=true_n_basis, coef_scale=0.6, noise_scale=0.15,
        prop_scale=1.5 * regime, seed=_seed("model", rep),
    )
    if not alternative:
        # The model class is unchanged, but both potential-outcome mean
        # coefficients are made equal before sampling.  This is a controlled
        # sharp null with known TATE == 0 and preserves covariate-driven noise.
        for d in range(sim.n_hom_dim):
            sim.Gamma[1][d] = sim.Gamma[0][d].copy()
    return sim.sample(n, rng=_seed("sample", rep, n, int(100 * regime), int(alternative)))


def _one_fit(sample, rep: int, n: int, regime: float, *, alternative: bool,
             n_calibration: int = N_CALIBRATION, n_bins: int = 8,
             n_basis: int = N_BASIS, propensity_estimator=None,
             propensity_feature_fn=None, include_forms: bool = False):
    fit = fit_dr(
        sample.observed, sample.tseq, n_basis=n_basis, n_folds=N_FOLDS,
        propensity_estimator=(propensity_estimator if propensity_estimator is not None else
                              RandomForestClassifier(
                                  n_estimators=100, min_samples_leaf=4,
                                  n_jobs=1, random_state=_seed("rf", rep, n))),
        random_state=_seed("fold", rep, n, int(100 * regime), int(alternative)),
        propensity_feature_fn=propensity_feature_fn,
    )
    # The strata are formed from the known design propensity in the oracle
    # benchmark.  This is deliberately stronger than estimated quantile
    # strata, and is the clean calibration reference.  In the cloud design
    # the fitted propensity strata are used instead and reported as such.
    strata_original = propensity_strata(sample.propensity, n_bins=n_bins)
    multiplier = multiplier_test(
        fit, n_draws=n_calibration, seed=_seed("mult", rep, n, int(100 * regime), int(alternative)),
    )
    permutation = stratified_permutation_test(
        fit, strata_original, n_perm=n_calibration,
        seed=_seed("perm", rep, n, int(100 * regime), int(alternative)),
    )
    equivalence = equivalence_test(
        fit, margin=0.25, n_draws=n_calibration,
        seed=_seed("equiv", rep, n, int(100 * regime), int(alternative)),
    )
    pos = positivity_diagnostics(fit)
    row = {
        "rep": int(rep), "n": int(n), "regime": float(regime),
        "alternative": bool(alternative),
        "multiplier_p": float(multiplier["pvalue"]),
        "permutation_p": float(permutation["pvalue"]),
        "statistic": float(multiplier["statistic"]),
        "estimate_sup": float(np.max(np.abs(multiplier["estimate"]))),
        "equivalence_p": float(equivalence["pvalue"]),
        "equivalence_reject_non_equivalence": bool(equivalence["reject_non_equivalence"]),
        "strata": "known_propensity",
        **{f"positivity_{k}": float(v) for k, v in pos.items()
           if isinstance(v, (int, float, np.integer, np.floating))},
    }
    # The raw sup statistic is the preregistered primary test.  The optional
    # diagnostics address the Phase 2 observation that n and statistic form
    # matter: L2, raw sup studentization, and L2 studentization use the same
    # fitted nuisances and strata, not fresh data.  They are off for the main
    # 500-replication fleet to keep the long run focused and bounded.
    if include_forms:
        for norm, studentized in (("l2", False), ("sup", True), ("l2", True)):
            tag = f"{norm}_{'studentized' if studentized else 'raw'}"
            m = multiplier_test(
                fit, n_draws=n_calibration, studentize=studentized, norm=norm,
                seed=_seed("mult", tag, rep, n, int(100 * regime), int(alternative)),
            )
            p = stratified_permutation_test(
                fit, strata_original, n_perm=n_calibration, studentize=studentized,
                norm=norm,
                seed=_seed("perm", tag, rep, n, int(100 * regime), int(alternative)),
            )
            row[f"multiplier_{tag}_p"] = float(m["pvalue"])
            row[f"permutation_{tag}_p"] = float(p["pvalue"])
    return row


def _oracle_replication(rep: int, sample_sizes=SAMPLE_SIZES,
                        regimes=REGIMES, n_calibration=N_CALIBRATION,
                        include_forms: bool = False):
    rows = []
    for n in sample_sizes:
        for regime in regimes:
            for alternative in (False, True):
                sample = _tri_sample(n, rep, regime, alternative=alternative)
                rows.append(_one_fit(sample, rep, n, regime,
                                     alternative=alternative,
                                     n_calibration=n_calibration,
                                     include_forms=include_forms))
    return rows


def _cloud_replication(rep: int, n: int = 100, regime: float = 1.0,
                       alternative: bool = False, n_calibration: int = 399,
                       include_forms: bool = False):
    dgp = CloudSampleDGP(
        n_per_group=n // 2, m=120, d_x=3,
        beta=np.array([-0.5, -0.1, 0.6]), prop_scale=1.5 * regime,
        group_effect=1 if alternative else 0,
        seed=_seed("cloud-model", rep, n),
    )
    sample = dgp.sample(rng=_seed("cloud-sample", rep, n, int(100 * regime), int(alternative)))
    phi, A, X = to_silhouette_sample(
        sample.clouds, sample.X, sample.A, filtration="alpha",
        homology_dims=(0, 1), interval=(0.0, 2.0), r=3.0, resolution=RESOLUTION,
    )
    # Propensity strata use the DGP's known design probability.  A production
    # analysis should pre-register how these are obtained when e(X) is unknown.
    observed = type("Observed", (), {
        "observed": (phi, A, X), "tseq": np.linspace(0.0, 2.0, RESOLUTION),
        "propensity": sample.propensity,
    })()
    return _one_fit(observed, rep, n, regime, alternative=alternative,
                    n_calibration=n_calibration, n_bins=8,
                    include_forms=include_forms)


def learner_sweep(rep: int, n: int = 200, regime: float = 1.0,
                  n_calibration: int = N_CALIBRATION,
                  include_forms: bool = False):
    """Task 3.3 benchmark: four propensity learners on one oracle design."""
    sample = _tri_sample(n, rep, regime, alternative=False)
    rows = []
    for name, estimator in propensity_learner_grid(seed=_seed("learner", rep)).items():
        # The grid is deterministic; cloning inside tcda_uq makes each fold
        # receive a fresh estimator.  Use the matching interaction features for
        # the logistic reference, while the flexible learners use raw X.
        feature_fn = _interaction_features if name == "logistic" else None
        row = _one_fit(sample, rep, n, regime, alternative=False,
                       n_calibration=n_calibration,
                       propensity_estimator=estimator,
                       propensity_feature_fn=feature_fn,
                       include_forms=include_forms)
        row["learner"] = name
        rows.append(row)
    return rows


def double_robustness_stress(rep: int, n: int = 200, regime: float = 1.0,
                             n_calibration: int = N_CALIBRATION):
    """Task 3.4: correct/misspecified e and mu configurations.

    The data-generating outcome mean has nine Fourier terms, while the main
    benchmark uses five.  The stress suite therefore has a deliberate
    outcome-regression misspecification when it fits three terms.  The
    propensity law contains two interactions, so raw-X logistic regression is
    the deliberate propensity misspecification; the interaction feature map is
    the correctly specified parametric reference.
    """
    sample = _tri_sample(n, rep, regime, alternative=False, true_n_basis=9)
    logistic = LogisticRegression(max_iter=2000, C=1e6, random_state=_seed("stress", rep))
    cases = {
        "both_correct": (9, logistic, _interaction_features),
        "propensity_misspecified": (9, logistic, None),
        "outcome_misspecified": (3, logistic, _interaction_features),
        "both_misspecified": (3, logistic, None),
    }
    rows = []
    for name, (n_basis, estimator, feature_fn) in cases.items():
        row = _one_fit(
            sample, rep, n, regime, alternative=False,
            n_calibration=n_calibration, n_basis=n_basis,
            propensity_estimator=estimator,
            propensity_feature_fn=feature_fn,
        )
        row["stress_case"] = name
        rows.append(row)
    return rows


def run_shard(shard_idx: int, reps_per_shard: int, *, design: str = "oracle",
              n_calibration: int = N_CALIBRATION, cloud_n: int = 100,
              include_forms: bool = False):
    os.makedirs(SHARDS, exist_ok=True)
    lo = int(shard_idx) * int(reps_per_shard)
    hi = lo + int(reps_per_shard)
    rows = []
    for rep in range(lo, hi):
        if design == "oracle":
            rows.extend(_oracle_replication(
                rep, n_calibration=n_calibration, include_forms=include_forms))
        elif design == "clouds":
            for alternative in (False, True):
                rows.append(_cloud_replication(
                    rep, n=cloud_n, regime=1.0, alternative=alternative,
                    n_calibration=n_calibration, include_forms=include_forms))
        elif design == "learners":
            rows.extend(learner_sweep(rep, n=cloud_n,
                                      n_calibration=n_calibration,
                                      include_forms=include_forms))
        elif design == "stress":
            rows.extend(double_robustness_stress(
                rep, n=cloud_n, n_calibration=n_calibration))
        else:
            raise ValueError("design must be oracle, clouds, learners, or stress")
    payload = {
        "design": design, "shard_idx": int(shard_idx),
        "reps": [lo, hi], "n_calibration": int(n_calibration),
        "rows": rows,
    }
    path = os.path.join(SHARDS, f"phase3_{design}_shard{shard_idx}.json")
    with open(path, "w") as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    return path


def _load_rows(pattern: str):
    import glob
    rows = []
    for path in sorted(glob.glob(pattern)):
        with open(path) as fh:
            rows.extend(json.load(fh)["rows"])
    return rows


def aggregate(design: str = "oracle", input_dir: str = SHARDS,
              output: str | None = None):
    rows = _load_rows(os.path.join(input_dir, f"phase3_{design}_shard*.json"))
    if not rows:
        raise FileNotFoundError(f"no Phase 3 {design} shard files in {input_dir}")
    grouped = defaultdict(list)
    for row in rows:
        key = (row.get("n"), row.get("regime"), row.get("alternative"), row.get("learner"))
        grouped[key].append(row)
    summary = []
    for (n, regime, alternative, learner), cells in sorted(grouped.items(), key=str):
        out = {
            "n": n, "regime": regime, "alternative": alternative,
            "learner": learner, "replications": len(cells),
        }
        for method in ("multiplier", "permutation"):
            vals = np.array([c[f"{method}_p"] for c in cells])
            out[f"{method}_rejection_rate"] = float(np.mean(vals < ALPHA))
            out[f"{method}_pvalue_mean"] = float(np.mean(vals))
            out[f"{method}_pvalue_mc_se"] = float(np.sqrt(max(out[f"{method}_rejection_rate"] *
                                                               (1 - out[f"{method}_rejection_rate"]), 0.0) /
                                                         len(vals)))
        if "equivalence_reject_non_equivalence" in cells[0]:
            out["equivalence_rejection_rate"] = float(np.mean([
                c["equivalence_reject_non_equivalence"] for c in cells]))
            out["equivalence_pvalue_mean"] = float(np.mean([
                c["equivalence_p"] for c in cells]))
        for key in ("positivity_min_pi", "positivity_max_pi",
                    "positivity_ess_treated", "positivity_ess_control"):
            if key in cells[0]:
                out[key + "_mean"] = float(np.mean([c[key] for c in cells]))
        summary.append(out)
    if output is None:
        output = os.path.join(RESULTS, f"phase3_{design}_summary.json")
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)
    with open(output, "w") as fh:
        json.dump({"design": design, "alpha": ALPHA, "rows": summary}, fh,
                  indent=2, sort_keys=True)
    print(json.dumps({"design": design, "rows": len(summary), "output": output}, indent=2))
    return output


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", choices=("shard", "aggregate", "smoke"), required=True)
    parser.add_argument("--design", choices=("oracle", "clouds", "learners", "stress"), default="oracle")
    parser.add_argument("--shard-idx", type=int, default=0)
    parser.add_argument("--reps-per-shard", type=int, default=10)
    parser.add_argument("--n-calibration", type=int, default=N_CALIBRATION)
    parser.add_argument("--cloud-n", type=int, default=100)
    parser.add_argument("--include-forms", action="store_true",
                        help="also record L2 and studentized calibration diagnostics")
    parser.add_argument("--input-dir", default=SHARDS)
    parser.add_argument("--output")
    args = parser.parse_args()
    if args.mode == "shard":
        path = run_shard(args.shard_idx, args.reps_per_shard,
                          design=args.design, n_calibration=args.n_calibration,
                          cloud_n=args.cloud_n, include_forms=args.include_forms)
        print(path)
    elif args.mode == "aggregate":
        aggregate(args.design, args.input_dir, args.output)
    else:
        # A bounded, end-to-end smoke that exercises every calibration path.
        rows = _oracle_replication(0, sample_sizes=(50,), regimes=(0.0,),
                                    n_calibration=39)
        print(json.dumps(rows, indent=2, sort_keys=True))
        print("Phase 3 smoke OK")


if __name__ == "__main__":
    main()


In [ ]:
import os, sys
sys.path.insert(0, '/content')
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'
os.makedirs('/tmp/mplconfig', exist_ok=True)
from experiments.phase3_dr_calibration import run_shard
from tda2s.tests.dr_outcome import fit_dr
from tcda_uq.datasets import TriOracleSimulation

sim = TriOracleSimulation(n_cov=3, n_hom_dim=2, resolution=16, n_basis=5, seed=0)
s = sim.sample(24, rng=0)
fit = fit_dr(s.observed, s.tseq, n_basis=5, n_folds=2, random_state=0)
assert fit.estimate.shape == (2, 16)
print('Phase 3 imports and cached-fit smoke passed')


In [ ]:
import time
SHARDS = [0, 1, 2, 3, 4]
REPS_PER_SHARD = 10
DESIGN = 'clouds'
N_CALIBRATION = 399
CLOUD_N = 100
from experiments.phase3_dr_calibration import run_shard

t0 = time.time()
for shard in SHARDS:
    path = run_shard(shard, REPS_PER_SHARD, design=DESIGN,
                     n_calibration=N_CALIBRATION, cloud_n=CLOUD_N)
    try:
        from google.colab import files
        files.download(path)
        print('Downloaded:', path)
    except Exception as exc:
        print('(Not on Colab / download skipped):', exc)
    print('completed shard', shard, 'elapsed seconds', round(time.time() - t0))
print('all shards done; files remain under /content/results/phase3_shards/')
